## Config and Import

In [10]:
# ============================================================
# 1. Imports
# ============================================================

import os
import json
import math
import random
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from PIL import Image

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch import Tensor
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from torch.utils.data import (
    Dataset,
    DataLoader,
    random_split,
)

from torchvision import transforms
from torchvision.ops import roi_align

from tqdm.auto import tqdm

In [11]:
# ============================================================
# 2. Environment Check
# ============================================================

print(f"PyTorch version:     {torch.__version__}")
print(f"CUDA available:      {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version:        {torch.version.cuda}")
    print(f"GPU name:            {torch.cuda.get_device_name(0)}")
    print(
        "GPU memory:          "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )
else:
    print("GPU is not enabled. Enable it from:")
    print("Runtime → Change runtime type → T4 GPU")

PyTorch version:     2.11.0+cpu
CUDA available:      False
GPU is not enabled. Enable it from:
Runtime → Change runtime type → T4 GPU


In [12]:
# ============================================================
# 3. Device
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Selected device:", DEVICE)

Selected device: cpu


In [13]:
# ============================================================
# 4. Reproducibility
# ============================================================

def set_seed(seed: int = 42) -> None:
    """
    Fix random seeds for reproducible experiments.
    """

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
set_seed(SEED)

print("Random seed:", SEED)

Random seed: 42


In [14]:
# ============================================================
# 5. Project Directories
# ============================================================

PROJECT_ROOT = Path("/content/relipose_hoi")

DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

for directory in [
    PROJECT_ROOT,
    DATA_DIR,
    CHECKPOINT_DIR,
    OUTPUT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Project root:  ", PROJECT_ROOT)
print("Data dir:      ", DATA_DIR)
print("Checkpoint dir:", CHECKPOINT_DIR)
print("Output dir:    ", OUTPUT_DIR)

Project root:   /content/relipose_hoi
Data dir:       /content/relipose_hoi/data
Checkpoint dir: /content/relipose_hoi/checkpoints
Output dir:     /content/relipose_hoi/outputs


In [93]:
# ============================================================
# 6. Global Configuration
# ============================================================

@dataclass
class Config:
    # Reproducibility
    seed: int = 42

    # Pose
    num_joints: int = 17

    # Feature dimensions
    local_joint_dim: int = 256
    pair_feature_dim: int = 512
    anatomy_feature_dim: int = 8
    joint_object_geometry_dim: int = 8
    object_embedding_dim: int = 64

    # Model
    model_dim: int = 256
    hidden_dim: int = 256
    joint_type_dim: int = 32
    num_attention_heads: int = 8
    dropout: float = 0.1

    # HICO-DET label space
    num_object_classes: int = 80
    num_verbs: int = 117
    num_hoi_classes: int = 600

    # Training
    batch_size: int = 16
    num_epochs: int = 20
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    num_workers: int = 2

    # Loss coefficients
    lambda_reliability: float = 1.0
    lambda_consistency: float = 0.1

    # Runtime
    device: str = str(DEVICE)

    #dataloader
    image_size: int = 640
    validation_ratio: float = 0.10
    horizontal_flip_probability: float = 0.5
    debug_max_samples: Optional[int] = None

    # Shared visual backbone
    backbone_name: str = "resnet50"
    pretrained_backbone: bool = True

    # Number of top ResNet stages that remain trainable.
    # 3 means layer2, layer3 and layer4 are trainable.
    trainable_backbone_stages: int = 3

    # All FPN feature maps will have this number of channels.
    fpn_out_channels: int = 256

    # Human RoI feature extraction
    human_roi_output_size: int = 14
    human_roi_sampling_ratio: int = 2

    # Integrated Sparse Pose Tokenizer
    pose_model_dim: int = 256
    pose_num_heads: int = 8
    pose_num_decoder_layers: int = 2
    pose_feedforward_dim: int = 1024
    pose_dropout: float = 0.1

    # Minimum positive value for predicted uncertainty
    pose_min_uncertainty: float = 1e-4

    # Pose-quality-guided sparse refinement
    pose_refinement_patch_size: int = 5

    # Patch radius relative to the normalized Human RoI.
    pose_refinement_radius: float = 0.12

    # A joint below this quality receives a stronger refinement gate.
    pose_refinement_quality_threshold: float = 0.50

    # Smaller values make the soft gate sharper.
    pose_refinement_temperature: float = 0.10

    # Maximum coordinate correction relative to the Human RoI.
    pose_refinement_max_offset: float = 0.15

    # Maximum multiplicative change of uncertainty in log space.
    pose_refinement_max_log_uncertainty_update: float = 1.0

    # Number of uncertain joints refined during inference.
    pose_refinement_topk: int = 6

    # Use true Top-K sparse computation during evaluation.
    pose_refinement_hard_topk_in_eval: bool = True

    pose_refinement_dropout: float = 0.10

CFG = Config()

## Dataset Prepration

In [17]:
# ============================================================
# 7.2. Install Dataset Download Utility
# ============================================================

!pip install -q gdown

In [18]:
import shutil
import subprocess
import tarfile

from typing import Iterable

import gdown

In [19]:
# ============================================================
# 7.3. HICO-DET Paths
# ============================================================

HICO_ROOT = DATA_DIR / "hico_20160224_det"

HICO_IMAGES_DIR = HICO_ROOT / "images"
HICO_TRAIN_IMAGES_DIR = HICO_IMAGES_DIR / "train2015"
HICO_TEST_IMAGES_DIR = HICO_IMAGES_DIR / "test2015"

HICO_ANNOTATIONS_DIR = HICO_ROOT / "annotations"

HICO_TRAIN_ANNOTATION = (
    HICO_ANNOTATIONS_DIR / "instances_train2015.json"
)

HICO_TEST_ANNOTATION = (
    HICO_ANNOTATIONS_DIR / "instances_test2015.json"
)

HICO_ARCHIVE_PATH = (
    DATA_DIR / "hico_20160224_det.tar.gz"
)

EXTERNAL_DIR = PROJECT_ROOT / "external"
HICODET_TOOLS_DIR = EXTERNAL_DIR / "hicodet"

EXTERNAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HICO_ANNOTATIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("HICO root:             ", HICO_ROOT)
print("Train images:          ", HICO_TRAIN_IMAGES_DIR)
print("Test images:           ", HICO_TEST_IMAGES_DIR)
print("Annotation directory:  ", HICO_ANNOTATIONS_DIR)

HICO root:              /content/relipose_hoi/data/hico_20160224_det
Train images:           /content/relipose_hoi/data/hico_20160224_det/images/train2015
Test images:            /content/relipose_hoi/data/hico_20160224_det/images/test2015
Annotation directory:   /content/relipose_hoi/data/hico_20160224_det/annotations


In [20]:
# ============================================================
# 7.4. Clone HICO-DET Annotation Utilities
# ============================================================

HICODET_REPOSITORY_URL = (
    "https://github.com/fredzzhang/hicodet.git"
)


def clone_repository(
    repository_url: str,
    destination: Path,
) -> None:
    """
    Clone a Git repository only when it does not already exist.
    """

    if (destination / ".git").exists():
        print(f"Repository already exists: {destination}")
        return

    if destination.exists():
        shutil.rmtree(destination)

    command = [
        "git",
        "clone",
        "--depth",
        "1",
        repository_url,
        str(destination),
    ]

    subprocess.run(
        command,
        check=True,
    )

    print(f"Repository cloned to: {destination}")


clone_repository(
    repository_url=HICODET_REPOSITORY_URL,
    destination=HICODET_TOOLS_DIR,
)

Repository cloned to: /content/relipose_hoi/external/hicodet


In [22]:
# ============================================================
# 7.5. Copy HICO-DET JSON Annotations
# ============================================================

annotation_files = [
    "instances_train2015.json",
    "instances_test2015.json",
    "coco80tohico80.json",
    "coco91tohico80.json",
]

for filename in annotation_files:
    source = HICODET_TOOLS_DIR / filename
    destination = HICO_ANNOTATIONS_DIR / filename

    shutil.copy2(
        source,
        destination,
    )

    print(f"Copied: {filename}")

Copied: instances_train2015.json
Copied: instances_test2015.json
Copied: coco80tohico80.json
Copied: coco91tohico80.json


In [23]:
assert HICO_TRAIN_ANNOTATION.exists()
assert HICO_TEST_ANNOTATION.exists()

print("Train annotation:", HICO_TRAIN_ANNOTATION)
print("Test annotation: ", HICO_TEST_ANNOTATION)

Train annotation: /content/relipose_hoi/data/hico_20160224_det/annotations/instances_train2015.json
Test annotation:  /content/relipose_hoi/data/hico_20160224_det/annotations/instances_test2015.json


In [24]:
# ============================================================
# 7.6. Download Official HICO-DET Archive
# ============================================================

# Current official Google Drive file ID
HICO_OFFICIAL_DRIVE_ID = (
    "1dUByzVzM6z1Oq4gENa1-t0FLhr0UtDaS"
)

# Mirror used by the public hicodet utility repository
HICO_MIRROR_DRIVE_ID = (
    "1QZcJmGVlF9f4h-XLWe9Gkmnmj2z1gSnk"
)


def archive_looks_valid(
    archive_path: Path,
    minimum_size_bytes: int = 1_000_000_000,
) -> bool:
    """
    Basic protection against incomplete Google Drive downloads.
    """

    return (
        archive_path.exists()
        and archive_path.is_file()
        and archive_path.stat().st_size >= minimum_size_bytes
    )


def download_hico_archive(
    output_path: Path,
) -> None:
    """
    Download HICO-DET from the official source, with one fallback mirror.
    """

    if archive_looks_valid(output_path):
        archive_size_gb = (
            output_path.stat().st_size / 1024**3
        )

        print(
            "Archive already exists:",
            f"{archive_size_gb:.2f} GB",
        )
        return

    drive_ids = [
        HICO_OFFICIAL_DRIVE_ID,
        HICO_MIRROR_DRIVE_ID,
    ]

    last_error = None

    for file_id in drive_ids:
        if output_path.exists():
            output_path.unlink()

        try:
            print(f"Trying Google Drive file: {file_id}")

            downloaded_path = gdown.download(
                id=file_id,
                output=str(output_path),
                quiet=False,
            )

            if (
                downloaded_path is not None
                and archive_looks_valid(output_path)
            ):
                archive_size_gb = (
                    output_path.stat().st_size / 1024**3
                )

                print(
                    "Download completed:",
                    f"{archive_size_gb:.2f} GB",
                )
                return

        except Exception as error:
            last_error = error
            print("Download attempt failed:", error)

    raise RuntimeError(
        "HICO-DET download failed from both sources."
    ) from last_error


download_hico_archive(
    output_path=HICO_ARCHIVE_PATH,
)

Trying Google Drive file: 1dUByzVzM6z1Oq4gENa1-t0FLhr0UtDaS


Downloading...
From (original): https://drive.google.com/uc?id=1dUByzVzM6z1Oq4gENa1-t0FLhr0UtDaS
From (redirected): https://drive.google.com/uc?id=1dUByzVzM6z1Oq4gENa1-t0FLhr0UtDaS&confirm=t&uuid=e024c365-4fc0-4cc0-aa5d-34efd790e58d
To: /content/relipose_hoi/data/hico_20160224_det.tar.gz
100%|██████████| 7.97G/7.97G [01:41<00:00, 78.5MB/s]

Download completed: 7.42 GB


In [25]:
# ============================================================
# 7.7. Extract HICO-DET
# ============================================================

def extract_hico_archive(
    archive_path: Path,
    destination_directory: Path,
    expected_train_directory: Path,
) -> None:
    """
    Extract HICO-DET only once.
    """

    if expected_train_directory.exists():
        print("HICO-DET has already been extracted.")
        return

    if not archive_path.exists():
        raise FileNotFoundError(
            f"Archive does not exist: {archive_path}"
        )

    print("Extracting HICO-DET...")

    subprocess.run(
        [
            "tar",
            "-xzf",
            str(archive_path),
            "-C",
            str(destination_directory),
        ],
        check=True,
    )

    if not expected_train_directory.exists():
        raise RuntimeError(
            "Extraction completed, but the expected "
            "train2015 directory was not found."
        )

    print("HICO-DET extraction completed.")


extract_hico_archive(
    archive_path=HICO_ARCHIVE_PATH,
    destination_directory=DATA_DIR,
    expected_train_directory=HICO_TRAIN_IMAGES_DIR,
)

Extracting HICO-DET...
HICO-DET extraction completed.


In [26]:
# ============================================================
# 7.8. Verify Image Directories
# ============================================================

if not HICO_TRAIN_IMAGES_DIR.exists():
    raise FileNotFoundError(
        f"Train directory not found: "
        f"{HICO_TRAIN_IMAGES_DIR}"
    )

if not HICO_TEST_IMAGES_DIR.exists():
    raise FileNotFoundError(
        f"Test directory not found: "
        f"{HICO_TEST_IMAGES_DIR}"
    )

train_image_files = sorted(
    HICO_TRAIN_IMAGES_DIR.glob("*.jpg")
)

test_image_files = sorted(
    HICO_TEST_IMAGES_DIR.glob("*.jpg")
)

print("Number of train images:", len(train_image_files))
print("Number of test images: ", len(test_image_files))

print()
print("First train image:", train_image_files[0])
print("First test image: ", test_image_files[0])

Number of train images: 38118
Number of test images:  9658

First train image: /content/relipose_hoi/data/hico_20160224_det/images/train2015/HICO_train2015_00000001.jpg
First test image:  /content/relipose_hoi/data/hico_20160224_det/images/test2015/HICO_test2015_00000001.jpg


In [27]:
# ============================================================
# 7.9. Inspect HICO-DET Annotation Metadata
# ============================================================

with open(
    HICO_TRAIN_ANNOTATION,
    mode="r",
    encoding="utf-8",
) as file:
    hico_train_metadata = json.load(file)

print("Top-level annotation keys:")

for key in hico_train_metadata.keys():
    print(" -", key)

Top-level annotation keys:
 - annotation
 - filenames
 - empty
 - objects
 - verbs
 - correspondence
 - size
 - rare
 - non_rare


In [28]:
# ============================================================
# 7.10. Verify HICO-DET Label Space
# ============================================================

num_filenames = len(
    hico_train_metadata["filenames"]
)

num_annotations = len(
    hico_train_metadata["annotation"]
)

num_empty_images = len(
    hico_train_metadata["empty"]
)

num_objects = len(
    hico_train_metadata["objects"]
)

num_verbs = len(
    hico_train_metadata["verbs"]
)

num_correspondences = len(
    hico_train_metadata["correspondence"]
)

print("Filenames:          ", num_filenames)
print("Annotation records: ", num_annotations)
print("Empty records:      ", num_empty_images)
print("Valid records:      ", num_filenames - num_empty_images)
print("Object classes:     ", num_objects)
print("Verb classes:       ", num_verbs)
print("HOI correspondences:", num_correspondences)

assert num_objects == CFG.num_object_classes
assert num_verbs == CFG.num_verbs
assert num_correspondences == CFG.num_hoi_classes

print("\nHICO-DET label-space validation passed.")

Filenames:           38118
Annotation records:  38118
Empty records:       485
Valid records:       37633
Object classes:      80
Verb classes:        117
HOI correspondences: 600

HICO-DET label-space validation passed.


In [34]:
# ============================================================
# 7.11. Inspect One Real HICO-DET Annotation
# ============================================================

empty_indices = set(
    hico_train_metadata["empty"]
)

sample_index = next(
    index
    for index in range(num_filenames)
    if index not in empty_indices
)

sample_filename = (
    hico_train_metadata["filenames"][sample_index]
)

sample_annotation = (
    hico_train_metadata["annotation"][sample_index]
)

sample_image_path = (
    HICO_TRAIN_IMAGES_DIR / sample_filename
)

print("Sample index:   ", sample_index)
print("Image filename: ", sample_filename)
print("Image exists:   ", sample_image_path.exists())
print("Annotation keys:", sample_annotation.keys())

print()
print("Human boxes:", len(sample_annotation["boxes_h"]))
print("Object boxes:", len(sample_annotation["boxes_o"]))
print("HOI labels: ", sample_annotation["hoi"])
print("Verb labels:", sample_annotation["verb"])
print("Objects:    ", sample_annotation["object"])

Sample index:    0
Image filename:  HICO_train2015_00000001.jpg
Image exists:    True
Annotation keys: dict_keys(['boxes_h', 'boxes_o', 'hoi', 'object', 'verb'])

Human boxes: 4
Object boxes: 4
HOI labels:  [152, 153, 154, 155]
Verb labels: [72, 76, 87, 98]
Objects:     [44, 44, 44, 44]


In [33]:
num_pairs = len(
    sample_annotation["boxes_h"]
)

assert len(sample_annotation["boxes_o"]) == num_pairs
assert len(sample_annotation["hoi"]) == num_pairs
assert len(sample_annotation["verb"]) == num_pairs
assert len(sample_annotation["object"]) == num_pairs

print(
    f"\nReal sample validation passed "
    f"with {num_pairs} human-object pairs."
)


Real sample validation passed with 4 human-object pairs.


## DataSet and Dataloader

In [36]:
CFG.debug_max_samples = 500

In [37]:
# ============================================================
# 9. Bounding-box Utilities
# ============================================================

def validate_xyxy_boxes(
    boxes: Tensor,
    name: str,
) -> None:
    """
    Validate bounding boxes in xyxy format.
    """

    if boxes.ndim != 2 or boxes.shape[-1] != 4:
        raise ValueError(
            f"{name} must have shape [N, 4], "
            f"received {tuple(boxes.shape)}."
        )

    if len(boxes) == 0:
        return

    if not torch.isfinite(boxes).all():
        raise ValueError(
            f"{name} contains NaN or Inf values."
        )

    widths = boxes[:, 2] - boxes[:, 0]
    heights = boxes[:, 3] - boxes[:, 1]

    if torch.any(widths <= 0):
        raise ValueError(
            f"{name} contains boxes with non-positive width."
        )

    if torch.any(heights <= 0):
        raise ValueError(
            f"{name} contains boxes with non-positive height."
        )


def clamp_boxes_to_image(
    boxes: Tensor,
    image_height: int,
    image_width: int,
) -> Tensor:
    """
    Clamp xyxy boxes to image boundaries.
    """

    if boxes.numel() == 0:
        return boxes

    boxes = boxes.clone()

    boxes[:, 0] = boxes[:, 0].clamp(
        min=0,
        max=max(image_width - 1, 0),
    )

    boxes[:, 2] = boxes[:, 2].clamp(
        min=0,
        max=max(image_width - 1, 0),
    )

    boxes[:, 1] = boxes[:, 1].clamp(
        min=0,
        max=max(image_height - 1, 0),
    )

    boxes[:, 3] = boxes[:, 3].clamp(
        min=0,
        max=max(image_height - 1, 0),
    )

    return boxes

In [38]:
# ============================================================
# 10. Convert Raw HICO Annotations
# ============================================================

def box_to_key(
    box: Sequence[float],
    decimal_places: int = 4,
) -> tuple[float, float, float, float]:
    """
    Convert a bounding box to a hashable key.

    Rounding protects against insignificant floating-point
    representation differences.
    """

    if len(box) != 4:
        raise ValueError(
            "A bounding box must contain four values."
        )

    return tuple(
        round(float(value), decimal_places)
        for value in box
    )


def convert_hico_annotation(
    annotation: Dict[str, Any],
    num_verbs: int,
    num_hoi_classes: int,
) -> Dict[str, Tensor]:
    """
    Convert repeated HICO-DET HOI rows into unique humans,
    unique objects and multi-label human-object pairs.
    """

    required_keys = {
        "boxes_h",
        "boxes_o",
        "verb",
        "object",
        "hoi",
    }

    missing_keys = required_keys - annotation.keys()

    if missing_keys:
        raise KeyError(
            f"Missing annotation fields: "
            f"{sorted(missing_keys)}"
        )

    boxes_h_raw = annotation["boxes_h"]
    boxes_o_raw = annotation["boxes_o"]
    verbs_raw = annotation["verb"]
    objects_raw = annotation["object"]
    hoi_raw = annotation["hoi"]

    number_of_rows = len(boxes_h_raw)

    field_lengths = {
        "boxes_o": len(boxes_o_raw),
        "verb": len(verbs_raw),
        "object": len(objects_raw),
        "hoi": len(hoi_raw),
    }

    for field_name, field_length in field_lengths.items():
        if field_length != number_of_rows:
            raise ValueError(
                f"Annotation field '{field_name}' has "
                f"{field_length} entries, but boxes_h has "
                f"{number_of_rows} entries."
            )

    human_key_to_index: Dict[
        tuple[float, float, float, float],
        int,
    ] = {}

    object_key_to_index: Dict[
        tuple[Any, ...],
        int,
    ] = {}

    pair_key_to_index: Dict[
        tuple[int, int],
        int,
    ] = {}

    unique_human_boxes: List[List[float]] = []
    unique_object_boxes: List[List[float]] = []
    unique_object_labels: List[int] = []

    pair_human_indices: List[int] = []
    pair_object_indices: List[int] = []

    pair_verb_targets: List[Tensor] = []
    pair_hoi_targets: List[Tensor] = []

    for row_index in range(number_of_rows):
        human_box = boxes_h_raw[row_index]
        object_box = boxes_o_raw[row_index]

        object_label = int(objects_raw[row_index])
        verb_label = int(verbs_raw[row_index])
        hoi_label = int(hoi_raw[row_index])

        if not 0 <= object_label < CFG.num_object_classes:
            raise ValueError(
                f"Invalid object label: {object_label}"
            )

        if not 0 <= verb_label < num_verbs:
            raise ValueError(
                f"Invalid verb label: {verb_label}"
            )

        if not 0 <= hoi_label < num_hoi_classes:
            raise ValueError(
                f"Invalid HOI label: {hoi_label}"
            )

        human_key = box_to_key(human_box)

        if human_key not in human_key_to_index:
            human_key_to_index[human_key] = (
                len(unique_human_boxes)
            )

            unique_human_boxes.append(
                [float(value) for value in human_box]
            )

        human_index = human_key_to_index[human_key]

        # Object category is included in the key.
        object_key = (
            *box_to_key(object_box),
            object_label,
        )

        if object_key not in object_key_to_index:
            object_key_to_index[object_key] = (
                len(unique_object_boxes)
            )

            unique_object_boxes.append(
                [float(value) for value in object_box]
            )

            unique_object_labels.append(object_label)

        object_index = object_key_to_index[object_key]

        pair_key = (
            human_index,
            object_index,
        )

        if pair_key not in pair_key_to_index:
            pair_index = len(pair_human_indices)

            pair_key_to_index[pair_key] = pair_index

            pair_human_indices.append(human_index)
            pair_object_indices.append(object_index)

            pair_verb_targets.append(
                torch.zeros(
                    num_verbs,
                    dtype=torch.float32,
                )
            )

            pair_hoi_targets.append(
                torch.zeros(
                    num_hoi_classes,
                    dtype=torch.float32,
                )
            )

        pair_index = pair_key_to_index[pair_key]

        pair_verb_targets[pair_index][verb_label] = 1.0
        pair_hoi_targets[pair_index][hoi_label] = 1.0

    if len(unique_human_boxes) == 0:
        human_boxes = torch.empty(
            (0, 4),
            dtype=torch.float32,
        )
    else:
        human_boxes = torch.tensor(
            unique_human_boxes,
            dtype=torch.float32,
        )

    if len(unique_object_boxes) == 0:
        object_boxes = torch.empty(
            (0, 4),
            dtype=torch.float32,
        )

        object_labels = torch.empty(
            (0,),
            dtype=torch.long,
        )
    else:
        object_boxes = torch.tensor(
            unique_object_boxes,
            dtype=torch.float32,
        )

        object_labels = torch.tensor(
            unique_object_labels,
            dtype=torch.long,
        )

    pair_human_indices_tensor = torch.tensor(
        pair_human_indices,
        dtype=torch.long,
    )

    pair_object_indices_tensor = torch.tensor(
        pair_object_indices,
        dtype=torch.long,
    )

    if len(pair_verb_targets) == 0:
        verb_targets = torch.empty(
            (0, num_verbs),
            dtype=torch.float32,
        )

        hoi_targets = torch.empty(
            (0, num_hoi_classes),
            dtype=torch.float32,
        )
    else:
        verb_targets = torch.stack(
            pair_verb_targets,
            dim=0,
        )

        hoi_targets = torch.stack(
            pair_hoi_targets,
            dim=0,
        )

    return {
        "human_boxes": human_boxes,
        "object_boxes": object_boxes,
        "object_labels": object_labels,
        "pair_human_indices": (
            pair_human_indices_tensor
        ),
        "pair_object_indices": (
            pair_object_indices_tensor
        ),
        "verb_targets": verb_targets,
        "hoi_targets": hoi_targets,
    }

In [39]:
# ============================================================
# 11. Test Annotation Conversion
# ============================================================

converted_annotation = convert_hico_annotation(
    annotation=sample_annotation,
    num_verbs=CFG.num_verbs,
    num_hoi_classes=CFG.num_hoi_classes,
)

for key, value in converted_annotation.items():
    print(
        f"{key:24s}",
        tuple(value.shape),
        value.dtype,
    )

print()
print(
    "Number of unique humans:",
    len(converted_annotation["human_boxes"]),
)

print(
    "Number of unique objects:",
    len(converted_annotation["object_boxes"]),
)

print(
    "Number of unique pairs:",
    len(converted_annotation["pair_human_indices"]),
)

print(
    "Number of positive verb labels:",
    int(converted_annotation["verb_targets"].sum()),
)

print(
    "Number of positive HOI labels:",
    int(converted_annotation["hoi_targets"].sum()),
)

human_boxes              (4, 4) torch.float32
object_boxes             (4, 4) torch.float32
object_labels            (4,) torch.int64
pair_human_indices       (4,) torch.int64
pair_object_indices      (4,) torch.int64
verb_targets             (4, 117) torch.float32
hoi_targets              (4, 600) torch.float32

Number of unique humans: 4
Number of unique objects: 4
Number of unique pairs: 4
Number of positive verb labels: 4
Number of positive HOI labels: 4


In [40]:
# ============================================================
# 12. Image and Box Transform
# ============================================================

import torchvision.transforms.functional as TF


class HICOImageTransform:
    """
    Resize an image while preserving aspect ratio, pad it to a
    square canvas, optionally flip it and normalize it.
    """

    def __init__(
        self,
        output_size: int,
        training: bool,
        horizontal_flip_probability: float = 0.5,
    ) -> None:
        self.output_size = output_size
        self.training = training

        self.horizontal_flip_probability = (
            horizontal_flip_probability
        )

        self.mean = [
            0.485,
            0.456,
            0.406,
        ]

        self.std = [
            0.229,
            0.224,
            0.225,
        ]

    @staticmethod
    def resize_boxes(
        boxes: Tensor,
        scale: float,
    ) -> Tensor:
        if boxes.numel() == 0:
            return boxes

        return boxes * scale

    @staticmethod
    def flip_boxes_horizontally(
        boxes: Tensor,
        image_width: int,
    ) -> Tensor:
        if boxes.numel() == 0:
            return boxes

        flipped_boxes = boxes.clone()

        old_x1 = boxes[:, 0].clone()
        old_x2 = boxes[:, 2].clone()

        flipped_boxes[:, 0] = (
            image_width - old_x2
        )

        flipped_boxes[:, 2] = (
            image_width - old_x1
        )

        return flipped_boxes

    def __call__(
        self,
        image: Image.Image,
        target: Dict[str, Tensor],
    ) -> tuple[Tensor, Dict[str, Tensor]]:

        if image.mode != "RGB":
            image = image.convert("RGB")

        original_width, original_height = image.size

        longest_side = max(
            original_height,
            original_width,
        )

        scale = (
            self.output_size / longest_side
        )

        resized_height = max(
            1,
            int(round(original_height * scale)),
        )

        resized_width = max(
            1,
            int(round(original_width * scale)),
        )

        image = TF.resize(
            image,
            size=[
                resized_height,
                resized_width,
            ],
            antialias=True,
        )

        target = {
            key: value.clone()
            if isinstance(value, Tensor)
            else value
            for key, value in target.items()
        }

        target["human_boxes"] = self.resize_boxes(
            target["human_boxes"],
            scale,
        )

        target["object_boxes"] = self.resize_boxes(
            target["object_boxes"],
            scale,
        )

        should_flip = (
            self.training
            and random.random()
            < self.horizontal_flip_probability
        )

        if should_flip:
            image = TF.hflip(image)

            target["human_boxes"] = (
                self.flip_boxes_horizontally(
                    target["human_boxes"],
                    resized_width,
                )
            )

            target["object_boxes"] = (
                self.flip_boxes_horizontally(
                    target["object_boxes"],
                    resized_width,
                )
            )

        padding_right = (
            self.output_size - resized_width
        )

        padding_bottom = (
            self.output_size - resized_height
        )

        image = TF.pad(
            image,
            padding=[
                0,
                0,
                padding_right,
                padding_bottom,
            ],
            fill=0,
        )

        image = TF.to_tensor(image)

        image = TF.normalize(
            image,
            mean=self.mean,
            std=self.std,
        )

        target["human_boxes"] = clamp_boxes_to_image(
            target["human_boxes"],
            image_height=self.output_size,
            image_width=self.output_size,
        )

        target["object_boxes"] = clamp_boxes_to_image(
            target["object_boxes"],
            image_height=self.output_size,
            image_width=self.output_size,
        )

        target["original_size"] = torch.tensor(
            [
                original_height,
                original_width,
            ],
            dtype=torch.long,
        )

        target["resized_size"] = torch.tensor(
            [
                resized_height,
                resized_width,
            ],
            dtype=torch.long,
        )

        target["input_size"] = torch.tensor(
            [
                self.output_size,
                self.output_size,
            ],
            dtype=torch.long,
        )

        target["resize_scale"] = torch.tensor(
            scale,
            dtype=torch.float32,
        )

        target["was_flipped"] = torch.tensor(
            should_flip,
            dtype=torch.bool,
        )

        return image, target

### Real custom dataset

In [41]:
# ============================================================
# 13. HICO-DET Custom Dataset
# ============================================================

class HICODetReliPoseDataset(Dataset):
    """
    Real HICO-DET dataset for the integrated ReliPose-HOI model.

    The dataset provides:
        - real image
        - unique human boxes
        - unique object boxes
        - object labels
        - human-object pair indices
        - multi-label verb targets
        - multi-label HOI targets

    It does not provide pose information. Pose is predicted
    inside the model.
    """

    def __init__(
        self,
        image_directory: str | Path,
        annotation_file: str | Path,
        transform: Optional[HICOImageTransform] = None,
        max_samples: Optional[int] = None,
    ) -> None:
        super().__init__()

        self.image_directory = Path(image_directory)
        self.annotation_file = Path(annotation_file)
        self.transform = transform

        if not self.image_directory.exists():
            raise FileNotFoundError(
                f"Image directory does not exist: "
                f"{self.image_directory}"
            )

        if not self.annotation_file.exists():
            raise FileNotFoundError(
                f"Annotation file does not exist: "
                f"{self.annotation_file}"
            )

        with open(
            self.annotation_file,
            mode="r",
            encoding="utf-8",
        ) as file:
            metadata = json.load(file)

        required_metadata_keys = {
            "filenames",
            "annotation",
            "empty",
            "correspondence",
            "objects",
            "verbs",
            "rare",
            "non_rare",
        }

        missing_keys = (
            required_metadata_keys - metadata.keys()
        )

        if missing_keys:
            raise KeyError(
                f"Missing metadata keys: "
                f"{sorted(missing_keys)}"
            )

        self.filenames = metadata["filenames"]
        self.annotations = metadata["annotation"]
        self.image_sizes = metadata.get(
            "size",
            None,
        )

        self.objects = metadata["objects"]
        self.verbs = metadata["verbs"]

        self.class_correspondence = (
            metadata["correspondence"]
        )

        self.rare_classes = metadata["rare"]
        self.non_rare_classes = metadata["non_rare"]

        empty_indices = set(metadata["empty"])

        self.valid_indices = [
            index
            for index in range(len(self.filenames))
            if index not in empty_indices
        ]

        if max_samples is not None:
            if max_samples <= 0:
                raise ValueError(
                    "max_samples must be positive."
                )

            self.valid_indices = (
                self.valid_indices[:max_samples]
            )

        if len(self.valid_indices) == 0:
            raise RuntimeError(
                "No valid HICO-DET samples were found."
            )

        if len(self.objects) != CFG.num_object_classes:
            raise ValueError(
                "Unexpected number of object classes: "
                f"{len(self.objects)}"
            )

        if len(self.verbs) != CFG.num_verbs:
            raise ValueError(
                "Unexpected number of verb classes: "
                f"{len(self.verbs)}"
            )

        if (
            len(self.class_correspondence)
            != CFG.num_hoi_classes
        ):
            raise ValueError(
                "Unexpected number of HOI classes: "
                f"{len(self.class_correspondence)}"
            )

    def __len__(self) -> int:
        return len(self.valid_indices)

    def __getitem__(
        self,
        index: int,
    ) -> tuple[Tensor, Dict[str, Tensor]]:

        if index < 0 or index >= len(self):
            raise IndexError(
                f"Dataset index {index} is out of range."
            )

        metadata_index = self.valid_indices[index]

        filename = self.filenames[metadata_index]

        image_path = (
            self.image_directory / filename
        )

        if not image_path.exists():
            raise FileNotFoundError(
                f"Image file does not exist: "
                f"{image_path}"
            )

        with Image.open(image_path) as pil_image:
            image = pil_image.convert("RGB")

        raw_annotation = (
            self.annotations[metadata_index]
        )

        target = convert_hico_annotation(
            annotation=raw_annotation,
            num_verbs=CFG.num_verbs,
            num_hoi_classes=CFG.num_hoi_classes,
        )

        original_width, original_height = image.size

        target["human_boxes"] = clamp_boxes_to_image(
            target["human_boxes"],
            image_height=original_height,
            image_width=original_width,
        )

        target["object_boxes"] = clamp_boxes_to_image(
            target["object_boxes"],
            image_height=original_height,
            image_width=original_width,
        )

        validate_xyxy_boxes(
            target["human_boxes"],
            name="human_boxes",
        )

        validate_xyxy_boxes(
            target["object_boxes"],
            name="object_boxes",
        )

        target["image_id"] = torch.tensor(
            metadata_index,
            dtype=torch.long,
        )

        target["dataset_index"] = torch.tensor(
            index,
            dtype=torch.long,
        )

        if self.transform is not None:
            image, target = self.transform(
                image,
                target,
            )
        else:
            image = TF.to_tensor(image)

        return image, target

In [42]:
# ============================================================
# 14. Train and Validation Transforms
# ============================================================

train_transform = HICOImageTransform(
    output_size=CFG.image_size,
    training=True,
    horizontal_flip_probability=(
        CFG.horizontal_flip_probability
    ),
)

validation_transform = HICOImageTransform(
    output_size=CFG.image_size,
    training=False,
    horizontal_flip_probability=0.0,
)

In [43]:
# ============================================================
# 15. Create Real HICO-DET Datasets
# ============================================================

full_train_dataset = HICODetReliPoseDataset(
    image_directory=HICO_TRAIN_IMAGES_DIR,
    annotation_file=HICO_TRAIN_ANNOTATION,
    transform=train_transform,
    max_samples=CFG.debug_max_samples,
)

full_validation_dataset = HICODetReliPoseDataset(
    image_directory=HICO_TRAIN_IMAGES_DIR,
    annotation_file=HICO_TRAIN_ANNOTATION,
    transform=validation_transform,
    max_samples=CFG.debug_max_samples,
)

test_dataset = HICODetReliPoseDataset(
    image_directory=HICO_TEST_IMAGES_DIR,
    annotation_file=HICO_TEST_ANNOTATION,
    transform=validation_transform,
    max_samples=CFG.debug_max_samples,
)

print("Available train images:", len(full_train_dataset))
print("Available test images: ", len(test_dataset))

Available train images: 500
Available test images:  500


In [44]:
# ============================================================
# 16. Reproducible Train/Validation Split
# ============================================================

number_of_samples = len(full_train_dataset)

number_of_validation_samples = max(
    1,
    int(
        round(
            number_of_samples
            * CFG.validation_ratio
        )
    ),
)

number_of_training_samples = (
    number_of_samples
    - number_of_validation_samples
)

split_generator = torch.Generator().manual_seed(
    CFG.seed
)

permutation = torch.randperm(
    number_of_samples,
    generator=split_generator,
).tolist()

validation_indices = permutation[
    :number_of_validation_samples
]

training_indices = permutation[
    number_of_validation_samples:
]

train_dataset = torch.utils.data.Subset(
    full_train_dataset,
    training_indices,
)

validation_dataset = torch.utils.data.Subset(
    full_validation_dataset,
    validation_indices,
)

print("Training images:  ", len(train_dataset))
print("Validation images:", len(validation_dataset))
print("Test images:      ", len(test_dataset))

assert (
    len(train_dataset)
    + len(validation_dataset)
    == number_of_samples
)

assert set(training_indices).isdisjoint(
    validation_indices
)

Training images:   450
Validation images: 50
Test images:       500


In [45]:
# ============================================================
# 17. Custom Collate Function
# ============================================================

def relipose_collate_fn(
    batch: List[
        tuple[
            Tensor,
            Dict[str, Tensor],
        ]
    ],
) -> tuple[
    Tensor,
    List[Dict[str, Tensor]],
]:
    """
    Stack fixed-size images while keeping variable-size
    annotation dictionaries in a list.
    """

    if len(batch) == 0:
        raise ValueError(
            "Cannot collate an empty batch."
        )

    images, targets = zip(*batch)

    images = torch.stack(
        images,
        dim=0,
    )

    return images, list(targets)

In [46]:
# ============================================================
# 18. DataLoaders
# ============================================================

DATALOADER_NUM_WORKERS = 0

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=DATALOADER_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=relipose_collate_fn,
    drop_last=False,
)

validation_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=DATALOADER_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=relipose_collate_fn,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=DATALOADER_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=relipose_collate_fn,
    drop_last=False,
)

print("Train batches:     ", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:      ", len(test_loader))

Train batches:      29
Validation batches: 4
Test batches:       32


In [47]:
# ============================================================
# 19. Inspect One Real Batch
# ============================================================

images, targets = next(iter(train_loader))

print("Images:", images.shape)
print("Images dtype:", images.dtype)

print()
print("Number of target dictionaries:", len(targets))

for sample_index, target in enumerate(targets[:3]):
    print(f"\nSample {sample_index}")
    print(
        "Human boxes:",
        tuple(target["human_boxes"].shape),
    )
    print(
        "Object boxes:",
        tuple(target["object_boxes"].shape),
    )
    print(
        "Object labels:",
        tuple(target["object_labels"].shape),
    )
    print(
        "Pair human indices:",
        tuple(target["pair_human_indices"].shape),
    )
    print(
        "Pair object indices:",
        tuple(target["pair_object_indices"].shape),
    )
    print(
        "Verb targets:",
        tuple(target["verb_targets"].shape),
    )
    print(
        "HOI targets:",
        tuple(target["hoi_targets"].shape),
    )

Images: torch.Size([16, 3, 640, 640])
Images dtype: torch.float32

Number of target dictionaries: 16

Sample 0
Human boxes: (3, 4)
Object boxes: (3, 4)
Object labels: (3,)
Pair human indices: (3,)
Pair object indices: (3,)
Verb targets: (3, 117)
HOI targets: (3, 600)

Sample 1
Human boxes: (5, 4)
Object boxes: (5, 4)
Object labels: (5,)
Pair human indices: (5,)
Pair object indices: (5,)
Verb targets: (5, 117)
HOI targets: (5, 600)

Sample 2
Human boxes: (1, 4)
Object boxes: (1, 4)
Object labels: (1,)
Pair human indices: (1,)
Pair object indices: (1,)
Verb targets: (1, 117)
HOI targets: (1, 600)


In [48]:
# ============================================================
# 21. Move Batch to Device
# ============================================================

def move_batch_to_device(
    images: Tensor,
    targets: List[Dict[str, Tensor]],
    device: torch.device,
) -> tuple[
    Tensor,
    List[Dict[str, Tensor]],
]:

    images = images.to(
        device=device,
        non_blocking=True,
    )

    device_targets = []

    for target in targets:
        device_target = {}

        for key, value in target.items():
            if isinstance(value, Tensor):
                device_target[key] = value.to(
                    device=device,
                    non_blocking=True,
                )
            else:
                device_target[key] = value

        device_targets.append(device_target)

    return images, device_targets

## Model Section

In [55]:
CFG = Config()

print("Backbone:", CFG.backbone_name)
print("Pretrained:", CFG.pretrained_backbone)
print(
    "Trainable backbone stages:",
    CFG.trainable_backbone_stages,
)
print("FPN output channels:", CFG.fpn_out_channels)

Backbone: resnet50
Pretrained: True
Trainable backbone stages: 3
FPN output channels: 256


In [56]:
# ============================================================
# 23. Backbone Imports
# ============================================================

from collections import OrderedDict

from torchvision.models import (
    ResNet50_Weights,
    resnet50,
)

from torchvision.ops import FeaturePyramidNetwork

### Backbone Model

In [57]:
# ============================================================
# 24. Shared ResNet-FPN Backbone
# ============================================================

class SharedResNetFPN(nn.Module):
    """
    Shared visual backbone for ReliPose-HOI.

    Input:
        images:
            Tensor[B, 3, H, W]

    Output:
        OrderedDict containing multi-scale FPN feature maps:
            "0": Tensor[B, D, H/4,  W/4]
            "1": Tensor[B, D, H/8,  W/8]
            "2": Tensor[B, D, H/16, W/16]
            "3": Tensor[B, D, H/32, W/32]

    The same feature maps will later be shared by:
        - human/object detection
        - human RoI feature extraction
        - Integrated Sparse Pose Tokenizer
        - human-object pair encoder
    """

    def __init__(
        self,
        out_channels: int = 256,
        pretrained: bool = True,
        trainable_stages: int = 3,
    ) -> None:
        super().__init__()

        if not 0 <= trainable_stages <= 5:
            raise ValueError(
                "trainable_stages must be between 0 and 5."
            )

        weights = (
            ResNet50_Weights.DEFAULT
            if pretrained
            else None
        )

        resnet = resnet50(
            weights=weights,
        )

        # Stem output stride: 4
        self.stem = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
        )

        # ResNet stages
        self.layer1 = resnet.layer1  # C2: 256 channels
        self.layer2 = resnet.layer2  # C3: 512 channels
        self.layer3 = resnet.layer3  # C4: 1024 channels
        self.layer4 = resnet.layer4  # C5: 2048 channels

        self.fpn = FeaturePyramidNetwork(
            in_channels_list=[
                256,
                512,
                1024,
                2048,
            ],
            out_channels=out_channels,
        )

        self.out_channels = out_channels
        self.trainable_stages = trainable_stages

        self._configure_trainable_stages()

    def _configure_trainable_stages(self) -> None:
        """
        Freeze lower ResNet stages and train only the selected
        highest-level stages.

        Stages are ordered as:
            stem, layer1, layer2, layer3, layer4
        """

        stages = [
            self.stem,
            self.layer1,
            self.layer2,
            self.layer3,
            self.layer4,
        ]

        # Freeze all ResNet stages first.
        for stage in stages:
            for parameter in stage.parameters():
                parameter.requires_grad = False

        # Unfreeze the selected highest stages.
        if self.trainable_stages > 0:
            stages_to_train = stages[
                -self.trainable_stages:
            ]

            for stage in stages_to_train:
                for parameter in stage.parameters():
                    parameter.requires_grad = True

        self._frozen_stages = [
            stage
            for stage in stages
            if not any(
                parameter.requires_grad
                for parameter in stage.parameters()
            )
        ]

    def train(
        self,
        mode: bool = True,
    ) -> "SharedResNetFPN":
        """
        Preserve frozen stages in evaluation mode so their
        BatchNorm running statistics are not modified.
        """

        super().train(mode)

        for stage in self._frozen_stages:
            stage.eval()

        return self

    def forward(
        self,
        images: Tensor,
    ) -> OrderedDict[str, Tensor]:

        if images.ndim != 4:
            raise ValueError(
                "images must have shape [B, C, H, W]."
            )

        if images.shape[1] != 3:
            raise ValueError(
                "images must have exactly three channels."
            )

        if not torch.isfinite(images).all():
            raise ValueError(
                "images contain NaN or Inf values."
            )

        # ResNet bottom-up pathway
        x = self.stem(images)

        c2 = self.layer1(x)
        c3 = self.layer2(c2)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)

        bottom_up_features = OrderedDict(
            [
                ("0", c2),
                ("1", c3),
                ("2", c4),
                ("3", c5),
            ]
        )

        # Top-down FPN pathway
        fpn_features = self.fpn(
            bottom_up_features
        )

        return fpn_features

In [58]:
# ============================================================
# 25. Instantiate Shared Backbone
# ============================================================

shared_backbone = SharedResNetFPN(
    out_channels=CFG.fpn_out_channels,
    pretrained=CFG.pretrained_backbone,
    trainable_stages=(
        CFG.trainable_backbone_stages
    ),
).to(DEVICE)

print(shared_backbone.__class__.__name__)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 107MB/s]


SharedResNetFPN


In [59]:
# ============================================================
# 26. Model Parameter Count
# ============================================================

def count_parameters(
    model: nn.Module,
) -> Dict[str, int]:

    total_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    frozen_parameters = (
        total_parameters
        - trainable_parameters
    )

    return {
        "total": total_parameters,
        "trainable": trainable_parameters,
        "frozen": frozen_parameters,
    }


backbone_parameter_count = count_parameters(
    shared_backbone
)

for name, value in backbone_parameter_count.items():
    print(f"{name:10s}: {value:,}")

total     : 26,852,416
trainable : 26,627,072
frozen    : 225,344


In [60]:
# ============================================================
# 27. Inspect Trainable Backbone Stages
# ============================================================

def module_is_trainable(
    module: nn.Module,
) -> bool:
    return any(
        parameter.requires_grad
        for parameter in module.parameters()
    )


backbone_stages = {
    "stem": shared_backbone.stem,
    "layer1": shared_backbone.layer1,
    "layer2": shared_backbone.layer2,
    "layer3": shared_backbone.layer3,
    "layer4": shared_backbone.layer4,
    "fpn": shared_backbone.fpn,
}

for stage_name, stage_module in backbone_stages.items():
    print(
        f"{stage_name:8s}: "
        f"trainable={module_is_trainable(stage_module)}"
    )

stem    : trainable=False
layer1  : trainable=False
layer2  : trainable=True
layer3  : trainable=True
layer4  : trainable=True
fpn     : trainable=True


#### Forward Test

In [61]:
# ============================================================
# 28. Backbone Forward Test
# ============================================================

shared_backbone.eval()

backbone_test_images = images[
    :min(2, images.shape[0])
].to(
    device=DEVICE,
    non_blocking=True,
)

with torch.no_grad():
    feature_maps = shared_backbone(
        backbone_test_images
    )

print(
    "Input shape:",
    tuple(backbone_test_images.shape),
)

print("\nFPN outputs:")

for level_name, feature_map in feature_maps.items():
    print(
        f"Level {level_name}: "
        f"shape={tuple(feature_map.shape)}, "
        f"dtype={feature_map.dtype}, "
        f"device={feature_map.device}"
    )

Input shape: (2, 3, 640, 640)

FPN outputs:
Level 0: shape=(2, 256, 160, 160), dtype=torch.float32, device=cpu
Level 1: shape=(2, 256, 80, 80), dtype=torch.float32, device=cpu
Level 2: shape=(2, 256, 40, 40), dtype=torch.float32, device=cpu
Level 3: shape=(2, 256, 20, 20), dtype=torch.float32, device=cpu


#### Backward test

In [63]:
# ============================================================
# 30. Backbone Backward Test
# ============================================================

shared_backbone.train()
shared_backbone.zero_grad(set_to_none=True)

backbone_train_test_images = images[:1].to(
    device=DEVICE,
    non_blocking=True,
)

training_feature_maps = shared_backbone(
    backbone_train_test_images
)

dummy_loss = sum(
    feature_map.square().mean()
    for feature_map in training_feature_maps.values()
)

dummy_loss.backward()

print(
    "Dummy backbone loss:",
    float(dummy_loss.detach().cpu()),
)

fpn_gradient_exists = any(
    parameter.grad is not None
    for parameter in shared_backbone.fpn.parameters()
    if parameter.requires_grad
)

layer4_gradient_exists = any(
    parameter.grad is not None
    for parameter in shared_backbone.layer4.parameters()
    if parameter.requires_grad
)

frozen_stem_has_gradient = any(
    parameter.grad is not None
    for parameter in shared_backbone.stem.parameters()
)

print("FPN received gradients:   ", fpn_gradient_exists)
print("Layer4 received gradients:", layer4_gradient_exists)
print("Frozen stem has gradient: ", frozen_stem_has_gradient)

assert fpn_gradient_exists
assert layer4_gradient_exists
assert not frozen_stem_has_gradient

print("Backbone backward test passed.")

shared_backbone.zero_grad(set_to_none=True)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Dummy backbone loss: 14.358654022216797
FPN received gradients:    True
Layer4 received gradients: True
Frozen stem has gradient:  False
Backbone backward test passed.


### Human RoIAlign Model

In [65]:
# ============================================================
# 32. Human RoIAlign Import
# ============================================================

from torchvision.ops import MultiScaleRoIAlign

In [66]:
# ============================================================
# 33. Prepare Human Boxes
# ============================================================

def prepare_human_boxes(
    targets: List[Dict[str, Tensor]],
    device: torch.device,
) -> List[Tensor]:
    """
    Extract human boxes from a batch of target dictionaries.

    Returns:
        List of length B.

        Each element has shape:
            [H_b, 4]

        Boxes are represented in image-level xyxy coordinates.
    """

    human_boxes = []

    for image_index, target in enumerate(targets):
        if "human_boxes" not in target:
            raise KeyError(
                f"Target {image_index} does not contain "
                f"'human_boxes'."
            )

        boxes = target["human_boxes"]

        if not isinstance(boxes, Tensor):
            raise TypeError(
                f"human_boxes[{image_index}] must be a Tensor."
            )

        if boxes.ndim != 2 or boxes.shape[-1] != 4:
            raise ValueError(
                f"human_boxes[{image_index}] must have shape "
                f"[H, 4], received {tuple(boxes.shape)}."
            )

        boxes = boxes.to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        )

        if boxes.numel() > 0:
            if not torch.isfinite(boxes).all():
                raise ValueError(
                    f"human_boxes[{image_index}] contains "
                    f"NaN or Inf."
                )

            widths = boxes[:, 2] - boxes[:, 0]
            heights = boxes[:, 3] - boxes[:, 1]

            if torch.any(widths <= 0):
                raise ValueError(
                    f"human_boxes[{image_index}] contains "
                    f"a box with non-positive width."
                )

            if torch.any(heights <= 0):
                raise ValueError(
                    f"human_boxes[{image_index}] contains "
                    f"a box with non-positive height."
                )

        human_boxes.append(boxes)

    return human_boxes

In [67]:
# ============================================================
# 34. Prepare Image Shapes
# ============================================================

def get_batch_image_shapes(
    images: Tensor,
) -> List[Tuple[int, int]]:
    """
    Return the network-input image shape for every batch item.

    MultiScaleRoIAlign expects:
        [(height_1, width_1), ..., (height_B, width_B)]
    """

    if images.ndim != 4:
        raise ValueError(
            "images must have shape [B, C, H, W]."
        )

    batch_size = images.shape[0]
    image_height = int(images.shape[-2])
    image_width = int(images.shape[-1])

    return [
        (image_height, image_width)
        for _ in range(batch_size)
    ]

In [68]:
image_shapes = get_batch_image_shapes(images)

print("Number of image shapes:", len(image_shapes))
print("First image shape:", image_shapes[0])

Number of image shapes: 16
First image shape: (640, 640)


In [69]:
# ============================================================
# 35. Human RoI Feature Extractor
# ============================================================

class HumanRoIFeatureExtractor(nn.Module):
    """
    Extract fixed-size feature maps for all humans in a batch.

    Inputs:
        feature_maps:
            OrderedDict[str, Tensor]
            Multi-scale FPN feature maps.

        human_boxes:
            List[Tensor]
            One [H_b, 4] xyxy tensor per image.

        image_shapes:
            List[(height, width)]
            Input image size for every batch item.

    Outputs:
        Dictionary containing:

        human_roi_features:
            [N_H, C, S, S]

        roi_to_image:
            [N_H]

        roi_to_human:
            [N_H]

        humans_per_image:
            [B]

        human_offsets:
            [B + 1]

    N_H is the total number of humans in the batch.
    """

    def __init__(
        self,
        feature_map_names: Sequence[str],
        output_size: int,
        sampling_ratio: int,
        out_channels: int,
    ) -> None:
        super().__init__()

        if output_size <= 0:
            raise ValueError(
                "output_size must be positive."
            )

        if sampling_ratio < -1:
            raise ValueError(
                "sampling_ratio must be greater than "
                "or equal to -1."
            )

        if out_channels <= 0:
            raise ValueError(
                "out_channels must be positive."
            )

        self.feature_map_names = list(
            feature_map_names
        )

        self.output_size = int(output_size)
        self.out_channels = int(out_channels)

        self.roi_pooler = MultiScaleRoIAlign(
            featmap_names=self.feature_map_names,
            output_size=self.output_size,
            sampling_ratio=sampling_ratio,
        )

    def _validate_inputs(
        self,
        feature_maps: Dict[str, Tensor],
        human_boxes: List[Tensor],
        image_shapes: List[Tuple[int, int]],
    ) -> None:

        if len(human_boxes) != len(image_shapes):
            raise ValueError(
                "The number of human-box tensors must match "
                "the number of image shapes."
            )

        missing_feature_maps = (
            set(self.feature_map_names)
            - set(feature_maps.keys())
        )

        if missing_feature_maps:
            raise KeyError(
                "Missing FPN feature levels: "
                f"{sorted(missing_feature_maps)}"
            )

        if len(feature_maps) == 0:
            raise ValueError(
                "feature_maps cannot be empty."
            )

        first_feature_map = next(
            iter(feature_maps.values())
        )

        batch_size = first_feature_map.shape[0]

        if batch_size != len(human_boxes):
            raise ValueError(
                "The FPN batch size does not match the "
                "number of human-box tensors."
            )

        for level_name in self.feature_map_names:
            feature_map = feature_maps[level_name]

            if feature_map.ndim != 4:
                raise ValueError(
                    f"Feature level {level_name} must have "
                    f"shape [B, C, H, W]."
                )

            if feature_map.shape[0] != batch_size:
                raise ValueError(
                    f"Invalid batch size at FPN level "
                    f"{level_name}."
                )

            if feature_map.shape[1] != self.out_channels:
                raise ValueError(
                    f"Expected {self.out_channels} channels "
                    f"at FPN level {level_name}, received "
                    f"{feature_map.shape[1]}."
                )

        for image_index, boxes in enumerate(human_boxes):
            if boxes.ndim != 2 or boxes.shape[-1] != 4:
                raise ValueError(
                    f"human_boxes[{image_index}] must have "
                    f"shape [H, 4]."
                )

    @staticmethod
    def _build_roi_metadata(
        human_boxes: List[Tensor],
        device: torch.device,
    ) -> Dict[str, Tensor]:
        """
        Construct mappings for the flattened RoI output.
        """

        humans_per_image = torch.tensor(
            [
                boxes.shape[0]
                for boxes in human_boxes
            ],
            dtype=torch.long,
            device=device,
        )

        roi_to_image_parts = []
        roi_to_human_parts = []

        for image_index, boxes in enumerate(human_boxes):
            number_of_humans = boxes.shape[0]

            if number_of_humans == 0:
                continue

            roi_to_image_parts.append(
                torch.full(
                    size=(number_of_humans,),
                    fill_value=image_index,
                    dtype=torch.long,
                    device=device,
                )
            )

            roi_to_human_parts.append(
                torch.arange(
                    number_of_humans,
                    dtype=torch.long,
                    device=device,
                )
            )

        if roi_to_image_parts:
            roi_to_image = torch.cat(
                roi_to_image_parts,
                dim=0,
            )

            roi_to_human = torch.cat(
                roi_to_human_parts,
                dim=0,
            )
        else:
            roi_to_image = torch.empty(
                0,
                dtype=torch.long,
                device=device,
            )

            roi_to_human = torch.empty(
                0,
                dtype=torch.long,
                device=device,
            )

        zero = torch.zeros(
            1,
            dtype=torch.long,
            device=device,
        )

        human_offsets = torch.cat(
            [
                zero,
                torch.cumsum(
                    humans_per_image,
                    dim=0,
                ),
            ],
            dim=0,
        )

        return {
            "roi_to_image": roi_to_image,
            "roi_to_human": roi_to_human,
            "humans_per_image": humans_per_image,
            "human_offsets": human_offsets,
        }

    def forward(
        self,
        feature_maps: Dict[str, Tensor],
        human_boxes: List[Tensor],
        image_shapes: List[Tuple[int, int]],
    ) -> Dict[str, Tensor]:

        self._validate_inputs(
            feature_maps=feature_maps,
            human_boxes=human_boxes,
            image_shapes=image_shapes,
        )

        first_feature_map = next(
            iter(feature_maps.values())
        )

        device = first_feature_map.device

        human_boxes = [
            boxes.to(
                device=device,
                dtype=torch.float32,
            )
            for boxes in human_boxes
        ]

        metadata = self._build_roi_metadata(
            human_boxes=human_boxes,
            device=device,
        )

        total_humans = int(
            metadata["humans_per_image"]
            .sum()
            .item()
        )

        if total_humans == 0:
            empty_features = torch.empty(
                (
                    0,
                    self.out_channels,
                    self.output_size,
                    self.output_size,
                ),
                dtype=first_feature_map.dtype,
                device=device,
            )

            return {
                "human_roi_features": empty_features,
                **metadata,
            }

        selected_feature_maps = OrderedDict(
            (
                level_name,
                feature_maps[level_name],
            )
            for level_name
            in self.feature_map_names
        )

        human_roi_features = self.roi_pooler(
            selected_feature_maps,
            human_boxes,
            image_shapes,
        )

        expected_shape = (
            total_humans,
            self.out_channels,
            self.output_size,
            self.output_size,
        )

        if human_roi_features.shape != expected_shape:
            raise RuntimeError(
                "Unexpected Human RoI feature shape. "
                f"Expected {expected_shape}, received "
                f"{tuple(human_roi_features.shape)}."
            )

        return {
            "human_roi_features": human_roi_features,
            **metadata,
        }

In [70]:
# ============================================================
# 36. Instantiate Human RoI Extractor
# ============================================================

human_roi_extractor = HumanRoIFeatureExtractor(
    feature_map_names=[
        "0",
        "1",
        "2",
        "3",
    ],
    output_size=CFG.human_roi_output_size,
    sampling_ratio=CFG.human_roi_sampling_ratio,
    out_channels=CFG.fpn_out_channels,
).to(DEVICE)

print(human_roi_extractor)

HumanRoIFeatureExtractor(
  (roi_pooler): MultiScaleRoIAlign(featmap_names=['0', '1', '2', '3'], output_size=(14, 14), sampling_ratio=2)
)


In [72]:
# ============================================================
# 37. Human RoI Forward Test
# ============================================================

shared_backbone.eval()
human_roi_extractor.eval()

number_of_test_images = min(
    2,
    images.shape[0],
)

roi_test_images = images[
    :number_of_test_images
].to(
    device=DEVICE,
    non_blocking=True,
)

roi_test_targets = targets[
    :number_of_test_images
]

roi_test_boxes = prepare_human_boxes(
    targets=roi_test_targets,
    device=DEVICE,
)

roi_test_shapes = get_batch_image_shapes(
    roi_test_images
)

with torch.no_grad():
    roi_test_feature_maps = shared_backbone(
        roi_test_images
    )

    human_roi_outputs = human_roi_extractor(
        feature_maps=roi_test_feature_maps,
        human_boxes=roi_test_boxes,
        image_shapes=roi_test_shapes,
    )

In [73]:
print(
    "Human RoI features:",
    tuple(
        human_roi_outputs[
            "human_roi_features"
        ].shape
    ),
)

print(
    "RoI to image:",
    tuple(
        human_roi_outputs[
            "roi_to_image"
        ].shape
    ),
)

print(
    "RoI to human:",
    tuple(
        human_roi_outputs[
            "roi_to_human"
        ].shape
    ),
)

print(
    "Humans per image:",
    human_roi_outputs[
        "humans_per_image"
    ].tolist(),
)

print(
    "Human offsets:",
    human_roi_outputs[
        "human_offsets"
    ].tolist(),
)

Human RoI features: (8, 256, 14, 14)
RoI to image: (8,)
RoI to human: (8,)
Humans per image: [3, 5]
Human offsets: [0, 3, 8]


In [74]:
# ============================================================
# 39. Inspect Flattened Human Mapping
# ============================================================

roi_to_image = human_roi_outputs[
    "roi_to_image"
].cpu()

roi_to_human = human_roi_outputs[
    "roi_to_human"
].cpu()

for roi_index in range(
    min(10, len(roi_to_image))
):
    print(
        f"Global RoI {roi_index:2d} -> "
        f"image={roi_to_image[roi_index].item()}, "
        f"human={roi_to_human[roi_index].item()}"
    )

Global RoI  0 -> image=0, human=0
Global RoI  1 -> image=0, human=1
Global RoI  2 -> image=0, human=2
Global RoI  3 -> image=1, human=0
Global RoI  4 -> image=1, human=1
Global RoI  5 -> image=1, human=2
Global RoI  6 -> image=1, human=3
Global RoI  7 -> image=1, human=4


In [75]:
# ============================================================
# 40. Local-to-Global Human Indices
# ============================================================

def build_global_pair_human_indices(
    targets: List[Dict[str, Tensor]],
    human_offsets: Tensor,
    device: torch.device,
) -> Tensor:
    """
    Convert per-image local human indices into flattened
    batch-level human indices.

    Example:
        humans per image = [2, 3]
        offsets = [0, 2, 5]

        image 0 local human 1 -> global human 1
        image 1 local human 1 -> global human 3
    """

    if human_offsets.shape != (
        len(targets) + 1,
    ):
        raise ValueError(
            "human_offsets has an invalid shape."
        )

    global_indices = []

    for image_index, target in enumerate(targets):
        local_indices = target[
            "pair_human_indices"
        ].to(
            device=device,
            dtype=torch.long,
        )

        if local_indices.numel() == 0:
            continue

        offset = human_offsets[image_index]

        global_indices.append(
            local_indices + offset
        )

    if not global_indices:
        return torch.empty(
            0,
            dtype=torch.long,
            device=device,
        )

    return torch.cat(
        global_indices,
        dim=0,
    )

In [76]:
global_pair_human_indices = (
    build_global_pair_human_indices(
        targets=roi_test_targets,
        human_offsets=human_roi_outputs[
            "human_offsets"
        ],
        device=DEVICE,
    )
)

print(
    "Global pair-human indices shape:",
    tuple(global_pair_human_indices.shape),
)

print(
    "First global pair-human indices:",
    global_pair_human_indices[
        :10
    ].tolist(),
)

Global pair-human indices shape: (8,)
First global pair-human indices: [0, 1, 2, 3, 4, 5, 6, 7]


In [77]:
# ============================================================
# 41. Human RoI Gradient Test (Backward)
# ============================================================

shared_backbone.train()
shared_backbone.zero_grad(set_to_none=True)

gradient_test_images = images[:1].to(
    device=DEVICE,
    non_blocking=True,
)

gradient_test_targets = targets[:1]

gradient_test_boxes = prepare_human_boxes(
    targets=gradient_test_targets,
    device=DEVICE,
)

gradient_test_shapes = get_batch_image_shapes(
    gradient_test_images
)

gradient_feature_maps = shared_backbone(
    gradient_test_images
)

gradient_roi_outputs = human_roi_extractor(
    feature_maps=gradient_feature_maps,
    human_boxes=gradient_test_boxes,
    image_shapes=gradient_test_shapes,
)

gradient_roi_features = gradient_roi_outputs[
    "human_roi_features"
]

if gradient_roi_features.shape[0] == 0:
    raise RuntimeError(
        "The gradient-test image contains no humans."
    )

roi_dummy_loss = (
    gradient_roi_features
    .square()
    .mean()
)

roi_dummy_loss.backward()

fpn_received_gradient = any(
    parameter.grad is not None
    for parameter
    in shared_backbone.fpn.parameters()
    if parameter.requires_grad
)

layer4_received_gradient = any(
    parameter.grad is not None
    for parameter
    in shared_backbone.layer4.parameters()
    if parameter.requires_grad
)

print(
    "Dummy RoI loss:",
    float(roi_dummy_loss.detach().cpu()),
)

print(
    "FPN received gradient:",
    fpn_received_gradient,
)

print(
    "Layer4 received gradient:",
    layer4_received_gradient,
)

assert fpn_received_gradient
assert layer4_received_gradient

print("Human RoI gradient test passed.")

shared_backbone.zero_grad(set_to_none=True)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Dummy RoI loss: 1.2436565160751343
FPN received gradient: True
Layer4 received gradient: True
Human RoI gradient test passed.


### Integrated Sparse Pose Tokenizer

In [81]:
# ============================================================
# 43. Human RoI Spatial Tokenizer
# ============================================================

class HumanRoISpatialTokenizer(nn.Module):
    """
    Convert Human RoI feature maps into spatial tokens.

    Input:
        human_roi_features:
            [N_H, C_in, S, S]

    Output:
        spatial_tokens:
            [N_H, S*S, D]
    """

    def __init__(
        self,
        input_channels: int,
        model_dim: int,
        roi_size: int,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        if input_channels <= 0:
            raise ValueError(
                "input_channels must be positive."
            )

        if model_dim <= 0:
            raise ValueError(
                "model_dim must be positive."
            )

        if roi_size <= 0:
            raise ValueError(
                "roi_size must be positive."
            )

        self.input_channels = input_channels
        self.model_dim = model_dim
        self.roi_size = roi_size
        self.num_spatial_tokens = roi_size * roi_size

        # Project FPN channels to the pose-token dimension.
        self.channel_projection = nn.Conv2d(
            in_channels=input_channels,
            out_channels=model_dim,
            kernel_size=1,
            stride=1,
            padding=0,
        )

        # Learnable positional embedding for the fixed 14×14 RoI.
        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.num_spatial_tokens,
                model_dim,
            )
        )

        self.token_norm = nn.LayerNorm(
            model_dim
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self._reset_parameters()

    def _reset_parameters(self) -> None:
        nn.init.xavier_uniform_(
            self.channel_projection.weight
        )

        if self.channel_projection.bias is not None:
            nn.init.zeros_(
                self.channel_projection.bias
            )

        nn.init.normal_(
            self.position_embedding,
            mean=0.0,
            std=0.02,
        )

    def forward(
        self,
        human_roi_features: Tensor,
    ) -> Tensor:

        if human_roi_features.ndim != 4:
            raise ValueError(
                "human_roi_features must have shape "
                "[N_H, C, H, W]."
            )

        number_of_humans = (
            human_roi_features.shape[0]
        )

        input_channels = (
            human_roi_features.shape[1]
        )

        roi_height = (
            human_roi_features.shape[2]
        )

        roi_width = (
            human_roi_features.shape[3]
        )

        if input_channels != self.input_channels:
            raise ValueError(
                f"Expected {self.input_channels} input "
                f"channels, received {input_channels}."
            )

        if (
            roi_height != self.roi_size
            or roi_width != self.roi_size
        ):
            raise ValueError(
                f"Expected RoI size "
                f"{self.roi_size}×{self.roi_size}, "
                f"received {roi_height}×{roi_width}."
            )

        if number_of_humans == 0:
            return torch.empty(
                (
                    0,
                    self.num_spatial_tokens,
                    self.model_dim,
                ),
                dtype=human_roi_features.dtype,
                device=human_roi_features.device,
            )

        projected_features = (
            self.channel_projection(
                human_roi_features
            )
        )

        # [N_H, D, S, S] -> [N_H, D, S*S]
        spatial_tokens = projected_features.flatten(
            start_dim=2
        )

        # [N_H, D, S*S] -> [N_H, S*S, D]
        spatial_tokens = spatial_tokens.transpose(
            1,
            2,
        ).contiguous()

        spatial_tokens = (
            spatial_tokens
            + self.position_embedding
        )

        spatial_tokens = self.token_norm(
            spatial_tokens
        )

        spatial_tokens = self.dropout(
            spatial_tokens
        )

        return spatial_tokens

In [82]:
# ============================================================
# 44. Test Human RoI Spatial Tokenizer
# ============================================================

spatial_tokenizer = HumanRoISpatialTokenizer(
    input_channels=CFG.fpn_out_channels,
    model_dim=CFG.pose_model_dim,
    roi_size=CFG.human_roi_output_size,
    dropout=CFG.pose_dropout,
).to(DEVICE)

spatial_tokenizer.eval()

test_human_roi_features = human_roi_outputs[
    "human_roi_features"
]

with torch.no_grad():
    test_spatial_tokens = spatial_tokenizer(
        test_human_roi_features
    )

print(
    "Human RoI features:",
    tuple(test_human_roi_features.shape),
)

print(
    "Spatial tokens:",
    tuple(test_spatial_tokens.shape),
)

Human RoI features: (8, 256, 14, 14)
Spatial tokens: (8, 196, 256)


In [83]:
# ============================================================
# 45. Integrated Sparse Pose Tokenizer
# ============================================================

class IntegratedSparsePoseTokenizer(nn.Module):
    """
    Initial integrated pose tokenizer for ReliPose-HOI.

    This module predicts an initial pose representation directly
    from Human RoI features, without using an external pose
    estimator.

    Input:
        human_roi_features:
            [N_H, C, S, S]

    Outputs:
        joint_tokens_initial:
            [N_H, K, D]

        joint_coordinates_coarse:
            [N_H, K, 2]
            Coordinates normalized to the Human RoI.

        joint_confidence:
            [N_H, K]

        joint_uncertainty:
            [N_H, K]

        coarse_quality:
            [N_H, K]
    """

    def __init__(
        self,
        input_channels: int,
        roi_size: int,
        num_joints: int,
        model_dim: int,
        num_heads: int,
        num_decoder_layers: int,
        feedforward_dim: int,
        dropout: float,
        minimum_uncertainty: float = 1e-4,
    ) -> None:
        super().__init__()

        if num_joints <= 0:
            raise ValueError(
                "num_joints must be positive."
            )

        if model_dim % num_heads != 0:
            raise ValueError(
                "model_dim must be divisible by "
                "num_heads."
            )

        if num_decoder_layers <= 0:
            raise ValueError(
                "num_decoder_layers must be positive."
            )

        if minimum_uncertainty <= 0:
            raise ValueError(
                "minimum_uncertainty must be positive."
            )

        self.input_channels = input_channels
        self.roi_size = roi_size
        self.num_joints = num_joints
        self.model_dim = model_dim

        self.minimum_uncertainty = (
            minimum_uncertainty
        )

        self.spatial_tokenizer = (
            HumanRoISpatialTokenizer(
                input_channels=input_channels,
                model_dim=model_dim,
                roi_size=roi_size,
                dropout=dropout,
            )
        )

        # Each embedding corresponds to one anatomical joint.
        self.joint_queries = nn.Embedding(
            num_embeddings=num_joints,
            embedding_dim=model_dim,
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=feedforward_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.joint_decoder = nn.TransformerDecoder(
            decoder_layer=decoder_layer,
            num_layers=num_decoder_layers,
            norm=nn.LayerNorm(model_dim),
        )

        self.joint_token_norm = nn.LayerNorm(
            model_dim
        )

        self.coordinate_head = nn.Sequential(
            nn.Linear(
                model_dim,
                model_dim,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                model_dim,
                2,
            ),
        )

        self.confidence_head = nn.Sequential(
            nn.Linear(
                model_dim,
                model_dim // 2,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                model_dim // 2,
                1,
            ),
        )

        self.uncertainty_head = nn.Sequential(
            nn.Linear(
                model_dim,
                model_dim // 2,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                model_dim // 2,
                1,
            ),
        )

        # Quality uses the token together with confidence
        # and uncertainty.
        self.quality_head = nn.Sequential(
            nn.Linear(
                model_dim + 2,
                model_dim // 2,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                model_dim // 2,
                1,
            ),
        )

        self._reset_parameters()

    def _reset_parameters(self) -> None:
        """
        Initialize independently cloned decoder layers and heads.
        """

        for name, parameter in self.named_parameters():
            if parameter.ndim > 1:
                nn.init.xavier_uniform_(
                    parameter
                )

            elif name.endswith("bias"):
                nn.init.zeros_(
                    parameter
                )

        nn.init.normal_(
            self.joint_queries.weight,
            mean=0.0,
            std=0.02,
        )

        # Keep initial coordinate predictions near the centre
        # of the Human RoI while retaining small differences
        # between joint queries.
        coordinate_output_layer = (
            self.coordinate_head[-1]
        )

        nn.init.normal_(
            coordinate_output_layer.weight,
            mean=0.0,
            std=1e-3,
        )

        nn.init.zeros_(
            coordinate_output_layer.bias
        )

        # Initial confidence is intentionally conservative.
        confidence_output_layer = (
            self.confidence_head[-1]
        )

        nn.init.normal_(
            confidence_output_layer.weight,
            mean=0.0,
            std=1e-3,
        )

        nn.init.constant_(
            confidence_output_layer.bias,
            -2.0,
        )

        # Softplus(-1.5) is approximately 0.20.
        uncertainty_output_layer = (
            self.uncertainty_head[-1]
        )

        nn.init.normal_(
            uncertainty_output_layer.weight,
            mean=0.0,
            std=1e-3,
        )

        nn.init.constant_(
            uncertainty_output_layer.bias,
            -1.5,
        )

    def _create_joint_queries(
        self,
        number_of_humans: int,
        device: torch.device,
    ) -> Tensor:
        """
        Expand K anatomical queries for every human.

        Output:
            [N_H, K, D]
        """

        joint_ids = torch.arange(
            self.num_joints,
            dtype=torch.long,
            device=device,
        )

        joint_queries = self.joint_queries(
            joint_ids
        )

        joint_queries = joint_queries.unsqueeze(
            0
        ).expand(
            number_of_humans,
            -1,
            -1,
        )

        return joint_queries

    def _empty_outputs(
        self,
        human_roi_features: Tensor,
    ) -> Dict[str, Tensor]:
        """
        Return shape-consistent outputs when no humans exist.
        """

        device = human_roi_features.device
        dtype = human_roi_features.dtype

        return {
            "joint_tokens_initial": torch.empty(
                (
                    0,
                    self.num_joints,
                    self.model_dim,
                ),
                dtype=dtype,
                device=device,
            ),

            "joint_coordinates_coarse": torch.empty(
                (
                    0,
                    self.num_joints,
                    2,
                ),
                dtype=dtype,
                device=device,
            ),

            "joint_confidence": torch.empty(
                (
                    0,
                    self.num_joints,
                ),
                dtype=dtype,
                device=device,
            ),

            "joint_uncertainty": torch.empty(
                (
                    0,
                    self.num_joints,
                ),
                dtype=dtype,
                device=device,
            ),

            "coarse_quality": torch.empty(
                (
                    0,
                    self.num_joints,
                ),
                dtype=dtype,
                device=device,
            ),
        }

    def forward(
        self,
        human_roi_features: Tensor,
    ) -> Dict[str, Tensor]:

        if human_roi_features.ndim != 4:
            raise ValueError(
                "human_roi_features must have shape "
                "[N_H, C, H, W]."
            )

        number_of_humans = (
            human_roi_features.shape[0]
        )

        if number_of_humans == 0:
            return self._empty_outputs(
                human_roi_features
            )

        spatial_tokens = self.spatial_tokenizer(
            human_roi_features
        )

        joint_queries = self._create_joint_queries(
            number_of_humans=number_of_humans,
            device=human_roi_features.device,
        )

        # Joint queries are the decoder target.
        # Human spatial tokens are the decoder memory.
        joint_tokens = self.joint_decoder(
            tgt=joint_queries,
            memory=spatial_tokens,
        )

        joint_tokens = self.joint_token_norm(
            joint_tokens
        )

        coordinate_logits = self.coordinate_head(
            joint_tokens
        )

        joint_coordinates_coarse = torch.sigmoid(
            coordinate_logits
        )

        confidence_logits = self.confidence_head(
            joint_tokens
        ).squeeze(-1)

        joint_confidence = torch.sigmoid(
            confidence_logits
        )

        uncertainty_logits = self.uncertainty_head(
            joint_tokens
        ).squeeze(-1)

        joint_uncertainty = (
            F.softplus(
                uncertainty_logits
            )
            + self.minimum_uncertainty
        )

        quality_input = torch.cat(
            [
                joint_tokens,
                joint_confidence.unsqueeze(-1),
                joint_uncertainty.unsqueeze(-1),
            ],
            dim=-1,
        )

        coarse_quality = torch.sigmoid(
            self.quality_head(
                quality_input
            ).squeeze(-1)
        )

        return {
            "joint_tokens_initial": joint_tokens,
            "joint_coordinates_coarse": (
                joint_coordinates_coarse
            ),
            "joint_confidence": joint_confidence,
            "joint_uncertainty": joint_uncertainty,
            "coarse_quality": coarse_quality,
        }

In [84]:
# ============================================================
# 46. Instantiate Integrated Sparse Pose Tokenizer
# ============================================================

pose_tokenizer = IntegratedSparsePoseTokenizer(
    input_channels=CFG.fpn_out_channels,
    roi_size=CFG.human_roi_output_size,
    num_joints=CFG.num_joints,
    model_dim=CFG.pose_model_dim,
    num_heads=CFG.pose_num_heads,
    num_decoder_layers=(
        CFG.pose_num_decoder_layers
    ),
    feedforward_dim=(
        CFG.pose_feedforward_dim
    ),
    dropout=CFG.pose_dropout,
    minimum_uncertainty=(
        CFG.pose_min_uncertainty
    ),
).to(DEVICE)

print(pose_tokenizer.__class__.__name__)

IntegratedSparsePoseTokenizer


In [85]:
pose_parameter_count = count_parameters(
    pose_tokenizer
)

for name, value in pose_parameter_count.items():
    print(
        f"{name:10s}: {value:,}"
    )

total     : 2,394,373
trainable : 2,394,373
frozen    : 0


In [86]:
# ============================================================
# 47. Pose Tokenizer Forward Test
# ============================================================

pose_tokenizer.eval()

test_human_roi_features = human_roi_outputs[
    "human_roi_features"
]

with torch.no_grad():
    pose_outputs = pose_tokenizer(
        test_human_roi_features
    )

for output_name, output_tensor in (
    pose_outputs.items()
):
    print(
        f"{output_name:28s}",
        tuple(output_tensor.shape),
        output_tensor.dtype,
    )

joint_tokens_initial         (8, 17, 256) torch.float32
joint_coordinates_coarse     (8, 17, 2) torch.float32
joint_confidence             (8, 17) torch.float32
joint_uncertainty            (8, 17) torch.float32
coarse_quality               (8, 17) torch.float32


In [87]:
# ============================================================
# 49. Inspect Initial Pose Output Statistics
# ============================================================

if pose_outputs[
    "joint_coordinates_coarse"
].numel() > 0:

    print(
        "Coordinate minimum:",
        float(
            pose_outputs[
                "joint_coordinates_coarse"
            ].min().cpu()
        ),
    )

    print(
        "Coordinate maximum:",
        float(
            pose_outputs[
                "joint_coordinates_coarse"
            ].max().cpu()
        ),
    )

    print(
        "Mean confidence:",
        float(
            pose_outputs[
                "joint_confidence"
            ].mean().cpu()
        ),
    )

    print(
        "Mean uncertainty:",
        float(
            pose_outputs[
                "joint_uncertainty"
            ].mean().cpu()
        ),
    )

    print(
        "Mean coarse quality:",
        float(
            pose_outputs[
                "coarse_quality"
            ].mean().cpu()
        ),
    )

Coordinate minimum: 0.49781304597854614
Coordinate maximum: 0.5021649599075317
Mean confidence: 0.11858585476875305
Mean uncertainty: 0.20045967400074005
Mean coarse quality: 0.4301178455352783


In [89]:
# ============================================================
# 48. Validate Pose Tokenizer Outputs
# ============================================================

def validate_pose_tokenizer_outputs(
    outputs: Dict[str, Tensor],
    number_of_humans: int,
    num_joints: int,
    model_dim: int,
) -> None:

    required_keys = {
        "joint_tokens_initial",
        "joint_coordinates_coarse",
        "joint_confidence",
        "joint_uncertainty",
        "coarse_quality",
    }

    missing_keys = required_keys - outputs.keys()

    if missing_keys:
        raise KeyError(
            f"Missing pose output keys: "
            f"{sorted(missing_keys)}"
        )

    joint_tokens = outputs[
        "joint_tokens_initial"
    ]

    coordinates = outputs[
        "joint_coordinates_coarse"
    ]

    confidence = outputs[
        "joint_confidence"
    ]

    uncertainty = outputs[
        "joint_uncertainty"
    ]

    quality = outputs[
        "coarse_quality"
    ]

    if joint_tokens.shape != (
        number_of_humans,
        num_joints,
        model_dim,
    ):
        raise ValueError(
            "Invalid joint token shape."
        )

    if coordinates.shape != (
        number_of_humans,
        num_joints,
        2,
    ):
        raise ValueError(
            "Invalid coordinate shape."
        )

    expected_scalar_shape = (
        number_of_humans,
        num_joints,
    )

    if confidence.shape != expected_scalar_shape:
        raise ValueError(
            "Invalid confidence shape."
        )

    if uncertainty.shape != expected_scalar_shape:
        raise ValueError(
            "Invalid uncertainty shape."
        )

    if quality.shape != expected_scalar_shape:
        raise ValueError(
            "Invalid coarse-quality shape."
        )

    for name, tensor in outputs.items():
        if not torch.isfinite(tensor).all():
            raise ValueError(
                f"{name} contains NaN or Inf."
            )

    if number_of_humans > 0:
        if torch.any(coordinates < 0):
            raise ValueError(
                "Coordinates below zero were found."
            )

        if torch.any(coordinates > 1):
            raise ValueError(
                "Coordinates above one were found."
            )

        if torch.any(confidence < 0):
            raise ValueError(
                "Confidence below zero was found."
            )

        if torch.any(confidence > 1):
            raise ValueError(
                "Confidence above one was found."
            )

        if torch.any(uncertainty <= 0):
            raise ValueError(
                "Uncertainty must be positive."
            )

        if torch.any(quality < 0):
            raise ValueError(
                "Quality below zero was found."
            )

        if torch.any(quality > 1):
            raise ValueError(
                "Quality above one was found."
            )

    print(
        "Pose tokenizer output validation passed."
    )


validate_pose_tokenizer_outputs(
    outputs=pose_outputs,
    number_of_humans=(
        test_human_roi_features.shape[0]
    ),
    num_joints=CFG.num_joints,
    model_dim=CFG.pose_model_dim,
)

# ============================================================
# 50. Empty-Human Test
# ============================================================

empty_human_features = torch.empty(
    (
        0,
        CFG.fpn_out_channels,
        CFG.human_roi_output_size,
        CFG.human_roi_output_size,
    ),
    dtype=torch.float32,
    device=DEVICE,
)

with torch.no_grad():
    empty_pose_outputs = pose_tokenizer(
        empty_human_features
    )

validate_pose_tokenizer_outputs(
    outputs=empty_pose_outputs,
    number_of_humans=0,
    num_joints=CFG.num_joints,
    model_dim=CFG.pose_model_dim,
)

for name, tensor in empty_pose_outputs.items():
    print(
        f"{name:28s}",
        tuple(tensor.shape),
    )

Pose tokenizer output validation passed.
Pose tokenizer output validation passed.
joint_tokens_initial         (0, 17, 256)
joint_coordinates_coarse     (0, 17, 2)
joint_confidence             (0, 17)
joint_uncertainty            (0, 17)
coarse_quality               (0, 17)


In [90]:
# ============================================================
# 51. End-to-End Pose Path Gradient Test
# ============================================================

shared_backbone.train()
pose_tokenizer.train()

shared_backbone.zero_grad(
    set_to_none=True
)

pose_tokenizer.zero_grad(
    set_to_none=True
)

pose_gradient_images = images[:1].to(
    device=DEVICE,
    non_blocking=True,
)

pose_gradient_targets = targets[:1]

pose_gradient_boxes = prepare_human_boxes(
    targets=pose_gradient_targets,
    device=DEVICE,
)

pose_gradient_shapes = get_batch_image_shapes(
    pose_gradient_images
)

pose_gradient_feature_maps = shared_backbone(
    pose_gradient_images
)

pose_gradient_roi_outputs = (
    human_roi_extractor(
        feature_maps=(
            pose_gradient_feature_maps
        ),
        human_boxes=pose_gradient_boxes,
        image_shapes=pose_gradient_shapes,
    )
)

pose_gradient_human_features = (
    pose_gradient_roi_outputs[
        "human_roi_features"
    ]
)

if (
    pose_gradient_human_features.shape[0]
    == 0
):
    raise RuntimeError(
        "The selected gradient-test image "
        "contains no humans."
    )

pose_gradient_outputs = pose_tokenizer(
    pose_gradient_human_features
)

pose_dummy_loss = (
    pose_gradient_outputs[
        "joint_tokens_initial"
    ].square().mean()

    + pose_gradient_outputs[
        "joint_coordinates_coarse"
    ].mean()

    + pose_gradient_outputs[
        "joint_confidence"
    ].mean()

    + pose_gradient_outputs[
        "joint_uncertainty"
    ].mean()

    + pose_gradient_outputs[
        "coarse_quality"
    ].mean()
)

pose_dummy_loss.backward()

pose_decoder_received_gradient = any(
    parameter.grad is not None
    for parameter
    in pose_tokenizer.joint_decoder.parameters()
    if parameter.requires_grad
)

joint_queries_received_gradient = (
    pose_tokenizer
    .joint_queries
    .weight
    .grad
    is not None
)

coordinate_head_received_gradient = any(
    parameter.grad is not None
    for parameter
    in pose_tokenizer.coordinate_head.parameters()
    if parameter.requires_grad
)

fpn_received_pose_gradient = any(
    parameter.grad is not None
    for parameter
    in shared_backbone.fpn.parameters()
    if parameter.requires_grad
)

print(
    "Dummy pose loss:",
    float(
        pose_dummy_loss
        .detach()
        .cpu()
    ),
)

print(
    "Pose decoder gradient:   ",
    pose_decoder_received_gradient,
)

print(
    "Joint queries gradient:  ",
    joint_queries_received_gradient,
)

print(
    "Coordinate head gradient:",
    coordinate_head_received_gradient,
)

print(
    "FPN received gradient:   ",
    fpn_received_pose_gradient,
)

assert pose_decoder_received_gradient
assert joint_queries_received_gradient
assert coordinate_head_received_gradient
assert fpn_received_pose_gradient

print(
    "End-to-end pose path gradient test passed."
)

shared_backbone.zero_grad(
    set_to_none=True
)

pose_tokenizer.zero_grad(
    set_to_none=True
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Dummy pose loss: 2.3763933181762695
Pose decoder gradient:    True
Joint queries gradient:   True
Coordinate head gradient: True
FPN received gradient:    True
End-to-end pose path gradient test passed.


### Pose-Quality-Guided Sparse Refinement

In [91]:
# ============================================================
# 53. Local Joint Patch Sampler
# ============================================================

class LocalJointPatchSampler(nn.Module):
    """
    Sample a small local feature patch around every joint.

    Inputs:
        human_roi_features:
            [N_H, C, H, W]

        joint_coordinates:
            [N_H, M, 2]
            Coordinates normalized to [0, 1] relative to
            the Human RoI.

    Output:
        patches:
            [N_H, M, C, P, P]

    M can be:
        - all K joints during soft training
        - only Top-M uncertain joints during inference
    """

    def __init__(
        self,
        patch_size: int,
        radius: float,
        padding_mode: str = "border",
        align_corners: bool = False,
    ) -> None:
        super().__init__()

        if patch_size <= 0:
            raise ValueError(
                "patch_size must be positive."
            )

        if patch_size % 2 == 0:
            raise ValueError(
                "patch_size must be odd."
            )

        if radius <= 0:
            raise ValueError(
                "radius must be positive."
            )

        if padding_mode not in {
            "zeros",
            "border",
            "reflection",
        }:
            raise ValueError(
                f"Unsupported padding mode: {padding_mode}"
            )

        self.patch_size = patch_size
        self.radius = radius
        self.padding_mode = padding_mode
        self.align_corners = align_corners

        # Offsets are defined in [0, 1] RoI coordinates.
        axis = torch.linspace(
            -radius,
            radius,
            steps=patch_size,
        )

        offset_y, offset_x = torch.meshgrid(
            axis,
            axis,
            indexing="ij",
        )

        roi_offsets = torch.stack(
            [
                offset_x,
                offset_y,
            ],
            dim=-1,
        )

        # grid_sample expects coordinates in [-1, 1].
        grid_offsets = roi_offsets * 2.0

        self.register_buffer(
            "grid_offsets",
            grid_offsets.view(
                1,
                1,
                patch_size,
                patch_size,
                2,
            ),
            persistent=False,
        )

    def forward(
        self,
        human_roi_features: Tensor,
        joint_coordinates: Tensor,
    ) -> Tensor:

        if human_roi_features.ndim != 4:
            raise ValueError(
                "human_roi_features must have shape "
                "[N_H, C, H, W]."
            )

        if (
            joint_coordinates.ndim != 3
            or joint_coordinates.shape[-1] != 2
        ):
            raise ValueError(
                "joint_coordinates must have shape "
                "[N_H, M, 2]."
            )

        number_of_humans = (
            human_roi_features.shape[0]
        )

        number_of_selected_joints = (
            joint_coordinates.shape[1]
        )

        channels = human_roi_features.shape[1]

        if (
            joint_coordinates.shape[0]
            != number_of_humans
        ):
            raise ValueError(
                "Human count differs between features "
                "and coordinates."
            )

        if (
            number_of_humans == 0
            or number_of_selected_joints == 0
        ):
            return torch.empty(
                (
                    number_of_humans,
                    number_of_selected_joints,
                    channels,
                    self.patch_size,
                    self.patch_size,
                ),
                dtype=human_roi_features.dtype,
                device=human_roi_features.device,
            )

        # [0, 1] -> [-1, 1]
        joint_centres = (
            joint_coordinates * 2.0 - 1.0
        )

        joint_centres = joint_centres.view(
            number_of_humans,
            number_of_selected_joints,
            1,
            1,
            2,
        )

        sampling_grid = (
            joint_centres
            + self.grid_offsets.to(
                dtype=joint_coordinates.dtype
            )
        )

        # Put all patches into one grid_sample call.
        sampling_grid = sampling_grid.reshape(
            number_of_humans,
            number_of_selected_joints
            * self.patch_size,
            self.patch_size,
            2,
        )

        sampled_features = F.grid_sample(
            input=human_roi_features,
            grid=sampling_grid,
            mode="bilinear",
            padding_mode=self.padding_mode,
            align_corners=self.align_corners,
        )

        # [N_H, C, M*P, P]
        #       ->
        # [N_H, M, C, P, P]
        sampled_features = sampled_features.reshape(
            number_of_humans,
            channels,
            number_of_selected_joints,
            self.patch_size,
            self.patch_size,
        )

        sampled_features = sampled_features.permute(
            0,
            2,
            1,
            3,
            4,
        ).contiguous()

        return sampled_features

In [94]:
# ============================================================
# 54. Test Local Joint Patch Sampler
# ============================================================

joint_patch_sampler = LocalJointPatchSampler(
    patch_size=CFG.pose_refinement_patch_size,
    radius=CFG.pose_refinement_radius,
).to(DEVICE)

test_local_patches = joint_patch_sampler(
    human_roi_features=(
        test_human_roi_features
    ),
    joint_coordinates=pose_outputs[
        "joint_coordinates_coarse"
    ],
)

print(
    "Human RoI features:",
    tuple(test_human_roi_features.shape),
)

print(
    "Joint coordinates:",
    tuple(
        pose_outputs[
            "joint_coordinates_coarse"
        ].shape
    ),
)

print(
    "Local joint patches:",
    tuple(test_local_patches.shape),
)

Human RoI features: (8, 256, 14, 14)
Joint coordinates: (8, 17, 2)
Local joint patches: (8, 17, 256, 5, 5)


In [95]:
# ============================================================
# 55. Local Joint Patch Encoder
# ============================================================

class LocalJointPatchEncoder(nn.Module):
    """
    Encode local joint patches into feature vectors.

    Input:
        patches:
            [N_H, M, C, P, P]

    Output:
        local_joint_features:
            [N_H, M, D]
    """

    def __init__(
        self,
        input_channels: int,
        model_dim: int,
        dropout: float,
    ) -> None:
        super().__init__()

        if input_channels <= 0:
            raise ValueError(
                "input_channels must be positive."
            )

        if model_dim <= 0:
            raise ValueError(
                "model_dim must be positive."
            )

        number_of_groups = 32

        while (
            number_of_groups > 1
            and model_dim % number_of_groups != 0
        ):
            number_of_groups //= 2

        self.input_channels = input_channels
        self.model_dim = model_dim

        self.encoder = nn.Sequential(
            nn.Conv2d(
                input_channels,
                model_dim,
                kernel_size=1,
                bias=False,
            ),
            nn.GroupNorm(
                number_of_groups,
                model_dim,
            ),
            nn.GELU(),

            nn.Conv2d(
                model_dim,
                model_dim,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.GroupNorm(
                number_of_groups,
                model_dim,
            ),
            nn.GELU(),

            nn.Dropout2d(dropout),

            nn.AdaptiveAvgPool2d(
                output_size=1,
            ),
        )

    def forward(
        self,
        patches: Tensor,
    ) -> Tensor:

        if patches.ndim != 5:
            raise ValueError(
                "patches must have shape "
                "[N_H, M, C, P, P]."
            )

        number_of_humans = patches.shape[0]
        number_of_joints = patches.shape[1]
        channels = patches.shape[2]
        patch_height = patches.shape[3]
        patch_width = patches.shape[4]

        if channels != self.input_channels:
            raise ValueError(
                f"Expected {self.input_channels} channels, "
                f"received {channels}."
            )

        if (
            number_of_humans == 0
            or number_of_joints == 0
        ):
            return torch.empty(
                (
                    number_of_humans,
                    number_of_joints,
                    self.model_dim,
                ),
                dtype=patches.dtype,
                device=patches.device,
            )

        flattened_patches = patches.reshape(
            number_of_humans
            * number_of_joints,
            channels,
            patch_height,
            patch_width,
        )

        encoded = self.encoder(
            flattened_patches
        )

        encoded = encoded.flatten(
            start_dim=1
        )

        encoded = encoded.reshape(
            number_of_humans,
            number_of_joints,
            self.model_dim,
        )

        return encoded

In [96]:
# ============================================================
# 56. Test Local Joint Patch Encoder
# ============================================================

local_patch_encoder = LocalJointPatchEncoder(
    input_channels=CFG.fpn_out_channels,
    model_dim=CFG.pose_model_dim,
    dropout=CFG.pose_refinement_dropout,
).to(DEVICE)

local_patch_encoder.eval()

with torch.no_grad():
    test_local_joint_features = (
        local_patch_encoder(
            test_local_patches
        )
    )

print(
    "Local joint features:",
    tuple(test_local_joint_features.shape),
)

assert test_local_joint_features.shape == (
    test_human_roi_features.shape[0],
    CFG.num_joints,
    CFG.pose_model_dim,
)

assert torch.isfinite(
    test_local_joint_features
).all()

print("Local patch encoder test passed.")

Local joint features: (8, 17, 256)
Local patch encoder test passed.


In [97]:
# ============================================================
# 57. Pose-Quality-Guided Sparse Refiner
# ============================================================

class PoseQualityGuidedSparseRefiner(nn.Module):
    """
    Refine uncertain joints using local Human RoI features.

    Training:
        Soft differentiable gate over all joints.

    Evaluation:
        Optionally refine only Top-K lowest-quality joints.

    The quality score rho is not the final Joint Reliability.
    """

    def __init__(
        self,
        feature_channels: int,
        model_dim: int,
        num_joints: int,
        patch_size: int,
        patch_radius: float,
        quality_threshold: float,
        temperature: float,
        maximum_coordinate_offset: float,
        maximum_log_uncertainty_update: float,
        inference_topk: int,
        hard_topk_in_eval: bool,
        minimum_uncertainty: float,
        dropout: float,
    ) -> None:
        super().__init__()

        if temperature <= 0:
            raise ValueError(
                "temperature must be positive."
            )

        if maximum_coordinate_offset <= 0:
            raise ValueError(
                "maximum_coordinate_offset must be positive."
            )

        if not 0 <= inference_topk <= num_joints:
            raise ValueError(
                "inference_topk must be between "
                "zero and num_joints."
            )

        self.model_dim = model_dim
        self.num_joints = num_joints

        self.quality_threshold = (
            quality_threshold
        )

        self.temperature = temperature

        self.maximum_coordinate_offset = (
            maximum_coordinate_offset
        )

        self.maximum_log_uncertainty_update = (
            maximum_log_uncertainty_update
        )

        self.inference_topk = inference_topk
        self.hard_topk_in_eval = hard_topk_in_eval

        self.minimum_uncertainty = (
            minimum_uncertainty
        )

        self.patch_sampler = (
            LocalJointPatchSampler(
                patch_size=patch_size,
                radius=patch_radius,
                padding_mode="border",
                align_corners=False,
            )
        )

        self.patch_encoder = (
            LocalJointPatchEncoder(
                input_channels=feature_channels,
                model_dim=model_dim,
                dropout=dropout,
            )
        )

        # token + local feature + xy + confidence
        # + uncertainty + coarse quality
        refinement_input_dim = (
            model_dim
            + model_dim
            + 2
            + 1
            + 1
            + 1
        )

        self.refinement_fusion = nn.Sequential(
            nn.Linear(
                refinement_input_dim,
                model_dim,
            ),
            nn.LayerNorm(model_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                model_dim,
                model_dim,
            ),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.coordinate_offset_head = nn.Linear(
            model_dim,
            2,
        )

        self.token_delta_head = nn.Linear(
            model_dim,
            model_dim,
        )

        self.confidence_delta_head = nn.Linear(
            model_dim,
            1,
        )

        self.uncertainty_delta_head = nn.Linear(
            model_dim,
            1,
        )

        self.final_token_norm = nn.LayerNorm(
            model_dim
        )

        self._reset_output_heads()

    def _reset_output_heads(self) -> None:
        """
        Start refinement as an identity transformation.
        """

        output_heads = [
            self.coordinate_offset_head,
            self.token_delta_head,
            self.confidence_delta_head,
            self.uncertainty_delta_head,
        ]

        for head in output_heads:
            nn.init.zeros_(head.weight)
            nn.init.zeros_(head.bias)

    def _soft_refinement_gate(
        self,
        coarse_quality: Tensor,
    ) -> Tensor:
        """
        Lower quality -> larger refinement gate.
        """

        return torch.sigmoid(
            (
                self.quality_threshold
                - coarse_quality
            )
            / self.temperature
        )

    @staticmethod
    def _gather_vectors(
        tensor: Tensor,
        indices: Tensor,
    ) -> Tensor:
        """
        Gather [N, K, D] using [N, M] indices.
        """

        expanded_indices = indices.unsqueeze(
            -1
        ).expand(
            -1,
            -1,
            tensor.shape[-1],
        )

        return torch.gather(
            tensor,
            dim=1,
            index=expanded_indices,
        )

    @staticmethod
    def _gather_scalars(
        tensor: Tensor,
        indices: Tensor,
    ) -> Tensor:
        """
        Gather [N, K] using [N, M] indices.
        """

        return torch.gather(
            tensor,
            dim=1,
            index=indices,
        )

    def _compute_refinement_candidates(
        self,
        human_roi_features: Tensor,
        joint_tokens: Tensor,
        joint_coordinates: Tensor,
        joint_confidence: Tensor,
        joint_uncertainty: Tensor,
        coarse_quality: Tensor,
    ) -> Dict[str, Tensor]:
        """
        Compute refinement predictions for M selected joints.
        """

        local_patches = self.patch_sampler(
            human_roi_features=(
                human_roi_features
            ),
            joint_coordinates=(
                joint_coordinates
            ),
        )

        local_joint_features = (
            self.patch_encoder(
                local_patches
            )
        )

        refinement_input = torch.cat(
            [
                joint_tokens,
                local_joint_features,
                joint_coordinates,
                joint_confidence.unsqueeze(-1),
                joint_uncertainty.unsqueeze(-1),
                coarse_quality.unsqueeze(-1),
            ],
            dim=-1,
        )

        hidden_features = self.refinement_fusion(
            refinement_input
        )

        coordinate_offset = torch.tanh(
            self.coordinate_offset_head(
                hidden_features
            )
        )

        coordinate_offset = (
            coordinate_offset
            * self.maximum_coordinate_offset
        )

        token_delta = self.token_delta_head(
            hidden_features
        )

        confidence_delta = (
            self.confidence_delta_head(
                hidden_features
            ).squeeze(-1)
        )

        uncertainty_delta = torch.tanh(
            self.uncertainty_delta_head(
                hidden_features
            ).squeeze(-1)
        )

        uncertainty_delta = (
            uncertainty_delta
            * self.maximum_log_uncertainty_update
        )

        safe_confidence = joint_confidence.clamp(
            min=1e-4,
            max=1.0 - 1e-4,
        )

        confidence_logits = torch.logit(
            safe_confidence
        )

        candidate_confidence = torch.sigmoid(
            confidence_logits
            + confidence_delta
        )

        candidate_uncertainty = (
            joint_uncertainty
            * torch.exp(
                uncertainty_delta
            )
        ).clamp_min(
            self.minimum_uncertainty
        )

        return {
            "local_joint_features": (
                local_joint_features
            ),
            "coordinate_offset": (
                coordinate_offset
            ),
            "token_delta": token_delta,
            "candidate_confidence": (
                candidate_confidence
            ),
            "candidate_uncertainty": (
                candidate_uncertainty
            ),
        }

    def _soft_forward(
        self,
        human_roi_features: Tensor,
        joint_tokens: Tensor,
        joint_coordinates: Tensor,
        joint_confidence: Tensor,
        joint_uncertainty: Tensor,
        coarse_quality: Tensor,
    ) -> Dict[str, Tensor]:

        candidates = (
            self._compute_refinement_candidates(
                human_roi_features=(
                    human_roi_features
                ),
                joint_tokens=joint_tokens,
                joint_coordinates=(
                    joint_coordinates
                ),
                joint_confidence=(
                    joint_confidence
                ),
                joint_uncertainty=(
                    joint_uncertainty
                ),
                coarse_quality=coarse_quality,
            )
        )

        refinement_gate = (
            self._soft_refinement_gate(
                coarse_quality
            )
        )

        vector_gate = refinement_gate.unsqueeze(
            -1
        )

        refined_coordinates = (
            joint_coordinates
            + vector_gate
            * candidates["coordinate_offset"]
        ).clamp(
            min=0.0,
            max=1.0,
        )

        refined_tokens = self.final_token_norm(
            joint_tokens
            + vector_gate
            * candidates["token_delta"]
        )

        refined_confidence = (
            joint_confidence
            + refinement_gate
            * (
                candidates[
                    "candidate_confidence"
                ]
                - joint_confidence
            )
        )

        refined_uncertainty = (
            joint_uncertainty
            + refinement_gate
            * (
                candidates[
                    "candidate_uncertainty"
                ]
                - joint_uncertainty
            )
        ).clamp_min(
            self.minimum_uncertainty
        )

        refinement_mask = (
            refinement_gate >= 0.5
        )

        return {
            "joint_tokens": refined_tokens,
            "joint_coordinates": (
                refined_coordinates
            ),
            "joint_confidence": (
                refined_confidence
            ),
            "joint_uncertainty": (
                refined_uncertainty
            ),
            "local_joint_features": (
                candidates[
                    "local_joint_features"
                ]
            ),
            "refinement_gate": (
                refinement_gate
            ),
            "refinement_mask": (
                refinement_mask
            ),
        }

    def _hard_topk_forward(
        self,
        human_roi_features: Tensor,
        joint_tokens: Tensor,
        joint_coordinates: Tensor,
        joint_confidence: Tensor,
        joint_uncertainty: Tensor,
        coarse_quality: Tensor,
    ) -> Dict[str, Tensor]:
        """
        Refine only the Top-K lowest-quality joints.
        """

        number_of_humans = (
            joint_tokens.shape[0]
        )

        number_to_refine = min(
            self.inference_topk,
            self.num_joints,
        )

        if number_to_refine == 0:
            return {
                "joint_tokens": joint_tokens,
                "joint_coordinates": (
                    joint_coordinates
                ),
                "joint_confidence": (
                    joint_confidence
                ),
                "joint_uncertainty": (
                    joint_uncertainty
                ),
                "local_joint_features": (
                    torch.zeros_like(
                        joint_tokens
                    )
                ),
                "refinement_gate": (
                    torch.zeros_like(
                        coarse_quality
                    )
                ),
                "refinement_mask": (
                    torch.zeros_like(
                        coarse_quality,
                        dtype=torch.bool,
                    )
                ),
            }

        # Smallest quality values are selected.
        selected_indices = torch.topk(
            coarse_quality,
            k=number_to_refine,
            dim=1,
            largest=False,
            sorted=False,
        ).indices

        selected_tokens = self._gather_vectors(
            joint_tokens,
            selected_indices,
        )

        selected_coordinates = (
            self._gather_vectors(
                joint_coordinates,
                selected_indices,
            )
        )

        selected_confidence = (
            self._gather_scalars(
                joint_confidence,
                selected_indices,
            )
        )

        selected_uncertainty = (
            self._gather_scalars(
                joint_uncertainty,
                selected_indices,
            )
        )

        selected_quality = (
            self._gather_scalars(
                coarse_quality,
                selected_indices,
            )
        )

        candidates = (
            self._compute_refinement_candidates(
                human_roi_features=(
                    human_roi_features
                ),
                joint_tokens=selected_tokens,
                joint_coordinates=(
                    selected_coordinates
                ),
                joint_confidence=(
                    selected_confidence
                ),
                joint_uncertainty=(
                    selected_uncertainty
                ),
                coarse_quality=(
                    selected_quality
                ),
            )
        )

        coordinate_index = (
            selected_indices
            .unsqueeze(-1)
            .expand(-1, -1, 2)
        )

        token_index = (
            selected_indices
            .unsqueeze(-1)
            .expand(
                -1,
                -1,
                self.model_dim,
            )
        )

        coordinate_update = torch.zeros_like(
            joint_coordinates
        ).scatter(
            dim=1,
            index=coordinate_index,
            src=candidates[
                "coordinate_offset"
            ],
        )

        token_update = torch.zeros_like(
            joint_tokens
        ).scatter(
            dim=1,
            index=token_index,
            src=candidates[
                "token_delta"
            ],
        )

        confidence_change = (
            candidates[
                "candidate_confidence"
            ]
            - selected_confidence
        )

        confidence_update = torch.zeros_like(
            joint_confidence
        ).scatter(
            dim=1,
            index=selected_indices,
            src=confidence_change,
        )

        uncertainty_change = (
            candidates[
                "candidate_uncertainty"
            ]
            - selected_uncertainty
        )

        uncertainty_update = torch.zeros_like(
            joint_uncertainty
        ).scatter(
            dim=1,
            index=selected_indices,
            src=uncertainty_change,
        )

        local_feature_update = torch.zeros_like(
            joint_tokens
        ).scatter(
            dim=1,
            index=token_index,
            src=candidates[
                "local_joint_features"
            ],
        )

        refinement_gate = torch.zeros_like(
            coarse_quality
        ).scatter(
            dim=1,
            index=selected_indices,
            src=torch.ones(
                (
                    number_of_humans,
                    number_to_refine,
                ),
                dtype=coarse_quality.dtype,
                device=coarse_quality.device,
            ),
        )

        refined_coordinates = (
            joint_coordinates
            + coordinate_update
        ).clamp(
            min=0.0,
            max=1.0,
        )

        refined_tokens = self.final_token_norm(
            joint_tokens
            + token_update
        )

        refined_confidence = (
            joint_confidence
            + confidence_update
        ).clamp(
            min=0.0,
            max=1.0,
        )

        refined_uncertainty = (
            joint_uncertainty
            + uncertainty_update
        ).clamp_min(
            self.minimum_uncertainty
        )

        return {
            "joint_tokens": refined_tokens,
            "joint_coordinates": (
                refined_coordinates
            ),
            "joint_confidence": (
                refined_confidence
            ),
            "joint_uncertainty": (
                refined_uncertainty
            ),
            "local_joint_features": (
                local_feature_update
            ),
            "refinement_gate": (
                refinement_gate
            ),
            "refinement_mask": (
                refinement_gate.bool()
            ),
        }

    def forward(
        self,
        human_roi_features: Tensor,
        coarse_pose_outputs: Dict[str, Tensor],
    ) -> Dict[str, Tensor]:

        required_keys = {
            "joint_tokens_initial",
            "joint_coordinates_coarse",
            "joint_confidence",
            "joint_uncertainty",
            "coarse_quality",
        }

        missing_keys = (
            required_keys
            - coarse_pose_outputs.keys()
        )

        if missing_keys:
            raise KeyError(
                f"Missing coarse pose outputs: "
                f"{sorted(missing_keys)}"
            )

        joint_tokens = coarse_pose_outputs[
            "joint_tokens_initial"
        ]

        joint_coordinates = (
            coarse_pose_outputs[
                "joint_coordinates_coarse"
            ]
        )

        joint_confidence = (
            coarse_pose_outputs[
                "joint_confidence"
            ]
        )

        joint_uncertainty = (
            coarse_pose_outputs[
                "joint_uncertainty"
            ]
        )

        coarse_quality = (
            coarse_pose_outputs[
                "coarse_quality"
            ]
        )

        number_of_humans = joint_tokens.shape[0]

        if number_of_humans == 0:
            return {
                "joint_tokens": joint_tokens,
                "joint_coordinates": (
                    joint_coordinates
                ),
                "joint_confidence": (
                    joint_confidence
                ),
                "joint_uncertainty": (
                    joint_uncertainty
                ),
                "local_joint_features": (
                    torch.empty_like(
                        joint_tokens
                    )
                ),
                "refinement_gate": (
                    torch.empty_like(
                        coarse_quality
                    )
                ),
                "refinement_mask": (
                    torch.empty_like(
                        coarse_quality,
                        dtype=torch.bool,
                    )
                ),
            }

        if (
            not self.training
            and self.hard_topk_in_eval
        ):
            return self._hard_topk_forward(
                human_roi_features=(
                    human_roi_features
                ),
                joint_tokens=joint_tokens,
                joint_coordinates=(
                    joint_coordinates
                ),
                joint_confidence=(
                    joint_confidence
                ),
                joint_uncertainty=(
                    joint_uncertainty
                ),
                coarse_quality=coarse_quality,
            )

        return self._soft_forward(
            human_roi_features=(
                human_roi_features
            ),
            joint_tokens=joint_tokens,
            joint_coordinates=(
                joint_coordinates
            ),
            joint_confidence=(
                joint_confidence
            ),
            joint_uncertainty=(
                joint_uncertainty
            ),
            coarse_quality=coarse_quality,
        )

In [98]:
# ============================================================
# 58. Instantiate Sparse Joint Refiner
# ============================================================

pose_refiner = PoseQualityGuidedSparseRefiner(
    feature_channels=CFG.fpn_out_channels,
    model_dim=CFG.pose_model_dim,
    num_joints=CFG.num_joints,

    patch_size=(
        CFG.pose_refinement_patch_size
    ),

    patch_radius=(
        CFG.pose_refinement_radius
    ),

    quality_threshold=(
        CFG.pose_refinement_quality_threshold
    ),

    temperature=(
        CFG.pose_refinement_temperature
    ),

    maximum_coordinate_offset=(
        CFG.pose_refinement_max_offset
    ),

    maximum_log_uncertainty_update=(
        CFG
        .pose_refinement_max_log_uncertainty_update
    ),

    inference_topk=(
        CFG.pose_refinement_topk
    ),

    hard_topk_in_eval=(
        CFG.pose_refinement_hard_topk_in_eval
    ),

    minimum_uncertainty=(
        CFG.pose_min_uncertainty
    ),

    dropout=(
        CFG.pose_refinement_dropout
    ),
).to(DEVICE)

print(
    pose_refiner.__class__.__name__
)

PoseQualityGuidedSparseRefiner


In [99]:
refiner_parameter_count = count_parameters(
    pose_refiner
)

for name, value in (
    refiner_parameter_count.items()
):
    print(
        f"{name:10s}: {value:,}"
    )

total     : 922,628
trainable : 922,628
frozen    : 0


In [100]:
# ============================================================
# 59. Soft Refinement Forward Test
# ============================================================

pose_refiner.train()

with torch.no_grad():
    soft_refinement_outputs = pose_refiner(
        human_roi_features=(
            test_human_roi_features
        ),
        coarse_pose_outputs=pose_outputs,
    )

for name, tensor in (
    soft_refinement_outputs.items()
):
    print(
        f"{name:24s}",
        tuple(tensor.shape),
        tensor.dtype,
    )

joint_tokens             (8, 17, 256) torch.float32
joint_coordinates        (8, 17, 2) torch.float32
joint_confidence         (8, 17) torch.float32
joint_uncertainty        (8, 17) torch.float32
local_joint_features     (8, 17, 256) torch.float32
refinement_gate          (8, 17) torch.float32
refinement_mask          (8, 17) torch.bool


In [101]:
if soft_refinement_outputs[
    "refinement_gate"
].numel() > 0:

    print(
        "Minimum gate:",
        float(
            soft_refinement_outputs[
                "refinement_gate"
            ].min().cpu()
        ),
    )

    print(
        "Maximum gate:",
        float(
            soft_refinement_outputs[
                "refinement_gate"
            ].max().cpu()
        ),
    )

    print(
        "Mean gate:",
        float(
            soft_refinement_outputs[
                "refinement_gate"
            ].mean().cpu()
        ),
    )

Minimum gate: 0.1767919957637787
Maximum gate: 0.9718705415725708
Mean gate: 0.6198351979255676


In [102]:
# ============================================================
# 60. Hard Top-K Refinement Test
# ============================================================

pose_refiner.eval()

with torch.no_grad():
    hard_refinement_outputs = pose_refiner(
        human_roi_features=(
            test_human_roi_features
        ),
        coarse_pose_outputs=pose_outputs,
    )

hard_mask = hard_refinement_outputs[
    "refinement_mask"
]

print(
    "Hard refinement mask shape:",
    tuple(hard_mask.shape),
)

if hard_mask.shape[0] > 0:
    print(
        "Refined joints per human:",
        hard_mask.sum(dim=1).tolist(),
    )

    expected_refined_joints = min(
        CFG.pose_refinement_topk,
        CFG.num_joints,
    )

    assert torch.all(
        hard_mask.sum(dim=1)
        == expected_refined_joints
    )

print("Hard Top-K refinement test passed.")

Hard refinement mask shape: (8, 17)
Refined joints per human: [6, 6, 6, 6, 6, 6, 6, 6]
Hard Top-K refinement test passed.


In [103]:
# ============================================================
# 61. Validate Sparse Refinement Outputs
# ============================================================

def validate_refinement_outputs(
    outputs: Dict[str, Tensor],
    number_of_humans: int,
    number_of_joints: int,
    model_dim: int,
) -> None:

    expected_shapes = {
        "joint_tokens": (
            number_of_humans,
            number_of_joints,
            model_dim,
        ),

        "joint_coordinates": (
            number_of_humans,
            number_of_joints,
            2,
        ),

        "joint_confidence": (
            number_of_humans,
            number_of_joints,
        ),

        "joint_uncertainty": (
            number_of_humans,
            number_of_joints,
        ),

        "local_joint_features": (
            number_of_humans,
            number_of_joints,
            model_dim,
        ),

        "refinement_gate": (
            number_of_humans,
            number_of_joints,
        ),

        "refinement_mask": (
            number_of_humans,
            number_of_joints,
        ),
    }

    for key, expected_shape in (
        expected_shapes.items()
    ):
        if key not in outputs:
            raise KeyError(
                f"Missing refinement output: {key}"
            )

        if outputs[key].shape != expected_shape:
            raise ValueError(
                f"Invalid shape for {key}. "
                f"Expected {expected_shape}, received "
                f"{tuple(outputs[key].shape)}."
            )

    for key, tensor in outputs.items():
        if tensor.dtype == torch.bool:
            continue

        if not torch.isfinite(tensor).all():
            raise ValueError(
                f"{key} contains NaN or Inf."
            )

    coordinates = outputs[
        "joint_coordinates"
    ]

    confidence = outputs[
        "joint_confidence"
    ]

    uncertainty = outputs[
        "joint_uncertainty"
    ]

    gate = outputs[
        "refinement_gate"
    ]

    if coordinates.numel() > 0:
        if torch.any(coordinates < 0):
            raise ValueError(
                "Refined coordinates below zero."
            )

        if torch.any(coordinates > 1):
            raise ValueError(
                "Refined coordinates above one."
            )

        if torch.any(confidence < 0):
            raise ValueError(
                "Confidence below zero."
            )

        if torch.any(confidence > 1):
            raise ValueError(
                "Confidence above one."
            )

        if torch.any(uncertainty <= 0):
            raise ValueError(
                "Uncertainty must be positive."
            )

        if torch.any(gate < 0):
            raise ValueError(
                "Refinement gate below zero."
            )

        if torch.any(gate > 1):
            raise ValueError(
                "Refinement gate above one."
            )

    print(
        "Sparse refinement validation passed."
    )

In [104]:
validate_refinement_outputs(
    outputs=soft_refinement_outputs,
    number_of_humans=(
        test_human_roi_features.shape[0]
    ),
    number_of_joints=CFG.num_joints,
    model_dim=CFG.pose_model_dim,
)

validate_refinement_outputs(
    outputs=hard_refinement_outputs,
    number_of_humans=(
        test_human_roi_features.shape[0]
    ),
    number_of_joints=CFG.num_joints,
    model_dim=CFG.pose_model_dim,
)

Sparse refinement validation passed.
Sparse refinement validation passed.


In [105]:
# ============================================================
# 62. Complete Integrated Pose Module
# ============================================================

class IntegratedPoseModule(nn.Module):
    """
    Complete internal pose branch of ReliPose-HOI.

    Human RoI features
        -> coarse sparse pose tokenizer
        -> pose-quality-guided sparse refinement
    """

    def __init__(
        self,
        pose_tokenizer: IntegratedSparsePoseTokenizer,
        pose_refiner: PoseQualityGuidedSparseRefiner,
    ) -> None:
        super().__init__()

        self.pose_tokenizer = pose_tokenizer
        self.pose_refiner = pose_refiner

    def forward(
        self,
        human_roi_features: Tensor,
    ) -> Dict[str, Tensor]:

        coarse_outputs = self.pose_tokenizer(
            human_roi_features
        )

        refined_outputs = self.pose_refiner(
            human_roi_features=(
                human_roi_features
            ),
            coarse_pose_outputs=(
                coarse_outputs
            ),
        )

        outputs = {
            "joint_tokens_initial": (
                coarse_outputs[
                    "joint_tokens_initial"
                ]
            ),

            "joint_coordinates_coarse": (
                coarse_outputs[
                    "joint_coordinates_coarse"
                ]
            ),

            "joint_confidence_coarse": (
                coarse_outputs[
                    "joint_confidence"
                ]
            ),

            "joint_uncertainty_coarse": (
                coarse_outputs[
                    "joint_uncertainty"
                ]
            ),

            "coarse_quality": (
                coarse_outputs[
                    "coarse_quality"
                ]
            ),

            **refined_outputs,
        }

        return outputs

In [106]:
integrated_pose_module = IntegratedPoseModule(
    pose_tokenizer=pose_tokenizer,
    pose_refiner=pose_refiner,
).to(DEVICE)

print(
    integrated_pose_module.__class__.__name__
)

IntegratedPoseModule


In [107]:
# ============================================================
# 63. Complete Integrated Pose Module Test
# ============================================================

integrated_pose_module.eval()

with torch.no_grad():
    integrated_pose_outputs = (
        integrated_pose_module(
            test_human_roi_features
        )
    )

for name, tensor in (
    integrated_pose_outputs.items()
):
    print(
        f"{name:28s}",
        tuple(tensor.shape),
    )

joint_tokens_initial         (8, 17, 256)
joint_coordinates_coarse     (8, 17, 2)
joint_confidence_coarse      (8, 17)
joint_uncertainty_coarse     (8, 17)
coarse_quality               (8, 17)
joint_tokens                 (8, 17, 256)
joint_coordinates            (8, 17, 2)
joint_confidence             (8, 17)
joint_uncertainty            (8, 17)
local_joint_features         (8, 17, 256)
refinement_gate              (8, 17)
refinement_mask              (8, 17)


In [109]:
# ============================================================
# 64. Full Sparse Pose Gradient Test
# ============================================================

def module_has_nonzero_gradient(
    module: nn.Module,
) -> bool:
    for parameter in module.parameters():
        if (
            parameter.requires_grad
            and parameter.grad is not None
            and torch.isfinite(
                parameter.grad
            ).all()
            and parameter.grad.abs().sum() > 0
        ):
            return True

    return False


shared_backbone.train()
integrated_pose_module.train()

shared_backbone.zero_grad(
    set_to_none=True
)

integrated_pose_module.zero_grad(
    set_to_none=True
)

refinement_gradient_images = (
    images[:1].to(
        device=DEVICE,
        non_blocking=True,
    )
)

refinement_gradient_targets = targets[:1]

refinement_gradient_boxes = (
    prepare_human_boxes(
        targets=refinement_gradient_targets,
        device=DEVICE,
    )
)

refinement_gradient_shapes = (
    get_batch_image_shapes(
        refinement_gradient_images
    )
)

refinement_feature_maps = shared_backbone(
    refinement_gradient_images
)

refinement_roi_outputs = (
    human_roi_extractor(
        feature_maps=refinement_feature_maps,
        human_boxes=refinement_gradient_boxes,
        image_shapes=refinement_gradient_shapes,
    )
)

refinement_human_features = (
    refinement_roi_outputs[
        "human_roi_features"
    ]
)

if refinement_human_features.shape[0] == 0:
    raise RuntimeError(
        "Gradient-test image contains no humans."
    )

refinement_outputs = integrated_pose_module(
    refinement_human_features
)

refinement_dummy_loss = (
    refinement_outputs[
        "joint_coordinates"
    ].mean()

    + refinement_outputs[
        "joint_confidence"
    ].mean()

    + refinement_outputs[
        "joint_uncertainty"
    ].mean()

    + refinement_outputs[
        "local_joint_features"
    ].abs().mean()

    + refinement_outputs[
        "joint_tokens"
    ][..., 0].mean()

    + refinement_outputs[
        "refinement_gate"
    ].mean()
)

refinement_dummy_loss.backward()

print(
    "Dummy refinement loss:",
    float(
        refinement_dummy_loss
        .detach()
        .cpu()
    ),
)

print(
    "Refinement fusion gradient:",
    module_has_nonzero_gradient(
        pose_refiner.refinement_fusion
    ),
)

print(
    "Patch encoder gradient:",
    module_has_nonzero_gradient(
        pose_refiner.patch_encoder
    ),
)

print(
    "Coordinate-offset gradient:",
    module_has_nonzero_gradient(
        pose_refiner.coordinate_offset_head
    ),
)

print(
    "Pose coordinate-head gradient:",
    module_has_nonzero_gradient(
        pose_tokenizer.coordinate_head
    ),
)

print(
    "Pose decoder gradient:",
    module_has_nonzero_gradient(
        pose_tokenizer.joint_decoder
    ),
)

print(
    "FPN gradient:",
    module_has_nonzero_gradient(
        shared_backbone.fpn
    ),
)

assert module_has_nonzero_gradient(
    pose_refiner.refinement_fusion
)

assert module_has_nonzero_gradient(
    pose_refiner.patch_encoder
)

assert module_has_nonzero_gradient(
    pose_refiner.coordinate_offset_head
)

assert module_has_nonzero_gradient(
    pose_tokenizer.coordinate_head
)

assert module_has_nonzero_gradient(
    pose_tokenizer.joint_decoder
)

assert module_has_nonzero_gradient(
    shared_backbone.fpn
)

print(
    "Full sparse-pose gradient test passed."
)

shared_backbone.zero_grad(
    set_to_none=True
)

integrated_pose_module.zero_grad(
    set_to_none=True
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Dummy refinement loss: 0.8854508996009827
Refinement fusion gradient: False
Patch encoder gradient: True
Coordinate-offset gradient: True
Pose coordinate-head gradient: True
Pose decoder gradient: True
FPN gradient: True


AssertionError: 

### d